<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_model/5_5_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **5_5_fine_tuning**

## **Introducción y Resumen**

### **Conclusiones del entrenamiento baseline**

A partir del entrenamiento del modelo baseline con batch_train = 256, se obtienen los siguientes resultados generales:

- RMSE en el orden de ≈ 0.002–0.003, con buena estabilidad entre folds.
- MAE bajo y consistente, alineado con el RMSE.
- R² entre 0.70 y 0.80, indicando una capacidad explicativa sólida.
- Directional Accuracy entre ≈ 0.81 y 0.84, un valor elevado para predicción direccional en series financieras intradía.
- El fold 5 presenta el mejor desempeño global, pero sin outliers que indiquen inestabilidad en los demás folds.

En conjunto, estos resultados confirman que el modelo baseline:

- Está correctamente entrenado,
- Generaliza de forma razonable,
- Y constituye una línea base robusta para iniciar el proceso de optimización de hiperparámetros.



### **Objetivos del coarse tuning**



El objetivo del proceso de coarse tuning será mejorar el desempeño del modelo respecto al baseline, priorizando:

- Reducir ligeramente RMSE y MAE.
- Incrementar R² en validación y test.
- Mantener o mejorar la Directional Accuracy.
- Reducir el gap entre validación y test, minimizando el sobreajuste.

### **Configuración fija del *baseline***


A partir de este punto, se establece una **configuración base fija**, común a todos los folds, que servirá como referencia para el proceso de *coarse tuning*.

**Parámetros de entrenamiento (`train_model`)**
- `lr = 3e-4`
- `weight_decay = 1e-4`
- `max_epochs = 50`
- `patience = 8`
- `grad_clip = 1.0`
- `use_amp = True`

**Parámetros del modelo y de los datos**
- `T = windows_size`
- `F = len(features_90)`
- `batch_train = 256`
- `pooling = "mean"` (en `TemporalPooling`)
- Arquitectura del *encoder* y del *head* (`d_model`, `n_heads`, `n_layers`, `dropout`, etc.) fijada según la configuración actual del baseline.

Sobre esta configuración base se llevará a cabo el **coarse tuning**, variando únicamente los hiperparámetros seleccionados, de manera controlada y comparable entre folds.

# Correr todo esto

## **0. Configuración del Entorno**


### 0.1. Instalación de librerías


In [1]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.9 MB/s eta 0:00:00


### 0.2. Importación de librerías


In [2]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna

In [3]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1
optuna: 4.6.0


### 0.3. Acceso a Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.4. Comprobación de uso de RAM

In [5]:
import psutil

def ram_usage():
    ram = psutil.virtual_memory()
    used = ram.used / (1024**3)
    free = ram.available / (1024**3)
    total = ram.total / (1024**3)

    print(f"RAM total:      {total:.2f} GB")
    print(f"RAM usada:      {used:.2f} GB")
    print(f"RAM disponible: {free:.2f} GB")

## **1. Carga de datos**

### **1.1. De datasets**

#### 1.1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [6]:
def load_data(data: str):

    data_path = f'{drive_path}/5_transformer_model/5_0_k_folds/fold_{fold}/{data}_{fold}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [7]:
#mnq_train = {}
#mnq_valid = {}
#mnq_test  = {}

#for k in k_folds:
#    print(f'Cargando datos de Fold {k}..')
#    mnq_train[k] = load_data(str(k), 'train')
#    mnq_valid[k] = load_data(str(k), 'valid')
#    mnq_test[k]  = load_data(str(k), 'test')

#### 1.1.2. Información de datasets


In [8]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [9]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

#### 1.1.3. Carga de listado de features por ventana de tiempo

In [10]:
import json
# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_90 = features_dict["features_to_90"]

In [11]:
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## **2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`**

In [12]:
#Lista de K folds
k_folds = [1, 2, 3, 4, 5]

### 2.0. Funciones

#### 2.0.1. Función para cargar ventanas

In [13]:
def load_windows_and_scaler(k: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz'
    path_valid  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz'
    path_test   = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/5_transformer_model/5_2_k_scaler/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    print(f'\tX_train_sc_{k} e y_train_{k} extraídos correctamente')
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    print(f'\tX_valid_sc_{k} e y_valid_{k} extraídos correctamente')
    X_test,  y_test  = data_test["X"],  data_test["y"]
    print(f'\tX_test_sc_{k} e y_test_{k} extraídos correctamente')

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### 2.0.2. Función para revisar información de ventanas

In [14]:
def xy_info(k, X_train, y_train, X_valid, y_valid, X_test, y_test, silent=False):
    import numpy as np
    import psutil

    if not silent:
        print(f"Información de {k}:")
        print("----------------------------------------")

    # Memoria total
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)

    def print_set_info(nombre, X, y):
        if silent:
            return  # No imprimir nada

        n_samples = X.shape[0]
        size_X_gb = X.nbytes / (1024 ** 3)
        size_y_gb = y.nbytes / (1024 ** 3)
        total_gb = size_X_gb + size_y_gb
        perc_ram = (total_gb / total_ram_gb) * 100
        y_flat = np.ravel(y)

        print(f"Set de {nombre}:")
        print(f"\t{n_samples} ventanas")
        print(f"\tTamaño X: {size_X_gb:.3f} GB")
        print(f"\tTamaño y: {size_y_gb:.6f} GB")
        print(f"\tTOTAL: {total_gb:.3f} GB → {perc_ram:.1f}% RAM\n")

    # Mostrar info solo si silent=False
    print_set_info("entrenamiento", X_train, y_train)
    print_set_info("validación",    X_valid, y_valid)
    print_set_info("testeo",        X_test,  y_test)

    # Pesos = cantidad de ventanas
    w_train = X_train.shape[0]
    w_valid = X_valid.shape[0]
    w_test  = X_test.shape[0]

    return w_train, w_valid, w_test


### 2.1 Carga de ventanas

In [15]:
#Para verificar el formato de lo guardado.
#for k in k_folds:
#    print(f'Fold {k}:')
#    print('\tTrain:\t', np.load(f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz').files)
#    print('\tValid:\t', np.load(f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz').files)
#    print('\tTest:\t',np.load(f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz').files)

In [16]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      1.48 GB
RAM disponible: 50.86 GB


In [17]:
# Diccionarios para almacenar datos escalados por fold
X_train_sc = {}
y_train_sc = {}
X_valid_sc = {}
y_valid_sc = {}
X_test_sc  = {}
y_test_sc  = {}
scalers    = {}

for k in k_folds:
    print(f'Fold {k}:')

    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(k)

    # Guardar todo en diccionarios
    X_train_sc[k] = X_train
    y_train_sc[k] = y_train

    X_valid_sc[k] = X_valid
    y_valid_sc[k] = y_valid

    X_test_sc[k]  = X_test
    y_test_sc[k]  = y_test

    scalers[k] = scaler

    print(f"  - Datos escalados cargados y almacenados en diccionarios.")
    print("-" * 40)

Fold 1:
	X_train_sc_1 e y_train_1 extraídos correctamente
	X_valid_sc_1 e y_valid_1 extraídos correctamente
	X_test_sc_1 e y_test_1 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 2:
	X_train_sc_2 e y_train_2 extraídos correctamente
	X_valid_sc_2 e y_valid_2 extraídos correctamente
	X_test_sc_2 e y_test_2 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 3:
	X_train_sc_3 e y_train_3 extraídos correctamente
	X_valid_sc_3 e y_valid_3 extraídos correctamente
	X_test_sc_3 e y_test_3 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 4:
	X_train_sc_4 e y_train_4 extraídos correctamente
	X_valid_sc_4 e y_valid_4 extraídos correctamente
	X_test_sc_4 e y_test_4 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
-------------

In [18]:
#Como accedeR:
#Xtr = X_train_sc[3]   # X_train del fold 3
#ytr = y_train_sc[3]

In [19]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      6.11 GB
RAM disponible: 46.22 GB


In [20]:
pesos_folds = {}

for k in k_folds:
    w_train, w_valid, w_test = xy_info(
        k,
        X_train_sc[k],
        y_train_sc[k],
        X_valid_sc[k],
        y_valid_sc[k],
        X_test_sc[k],
        y_test_sc[k],
        silent=True   # evita imprimir
    )

    pesos_folds[k] = {
        "w_train": w_train,
        "w_valid": w_valid,
        "w_test":  w_test,
    }


In [21]:
pesos_folds

{1: {'w_train': 124279, 'w_valid': 24898, 'w_test': 27852},
 2: {'w_train': 149177, 'w_valid': 24898, 'w_test': 27852},
 3: {'w_train': 174075, 'w_valid': 24898, 'w_test': 27852},
 4: {'w_train': 198973, 'w_valid': 24898, 'w_test': 27852},
 5: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852}}

In [22]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      6.11 GB
RAM disponible: 46.22 GB


## **3. Dataset de Métricas**

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [23]:
drive_path

'/content/drive/MyDrive/neural_profit'

In [24]:
def load_metrics(subcarpeta: str, data: str):
    data_path = f'{drive_path}/5_transformer_model/{subcarpeta}/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [25]:
def metrics_verify(subcarpeta: str, data: str) -> bool:
    data_path = f'{drive_path}/5_transformer_model/{subcarpeta}/{data}.parquet'
    return os.path.exists(data_path)


In [26]:
def load_or_create_metrics (subcarpeta: str, data:str):
  if metrics_verify(subcarpeta, data):
      print(f"Las métricas existen y son almacenadas en {data[2:len(data)]}")
      model_metrics = load_metrics(subcarpeta, data)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[2:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [27]:
baseline_folds_metrics, flag_baseline_folds_metrics = load_or_create_metrics("5_3_baseline_model", "0_baseline_folds_metrics")
baseline_metrics, flag_baseline_metrics = load_or_create_metrics("5_3_baseline_model", "1_baseline_metrics")

Las métricas existen y son almacenadas en baseline_folds_metrics
Las métricas existen y son almacenadas en baseline_metrics


In [28]:
coarse_folds_metrics, flag_coarse_folds_metrics = load_or_create_metrics("5_4_coarse_tuning", "0_coarse_folds_metrics")
coarse_metrics, flag_coarse_metrics = load_or_create_metrics("5_4_coarse_tuning", "1_coarse_metrics")

Las métricas existen y son almacenadas en coarse_folds_metrics
Las métricas existen y son almacenadas en coarse_metrics


In [29]:
fine_folds_metrics, flag_fine_folds_metrics = load_or_create_metrics("5_5_fine_tuning", "0_fine_folds_metrics")
fine_metrics, flag_fine_metrics = load_or_create_metrics("5_5_fine_tuning", "1_fine_metrics")

Las métricas no existen. Se crea el dataset fine_folds_metrics para almacenar las métricas
Las métricas no existen. Se crea el dataset fine_metrics para almacenar las métricas


### 3.2. Función para guardar métricas

In [30]:
def save_metrics (metrics,  subcarpeta: str, metrics_name: str):
  metrics_path = f"{drive_path}/5_transformer_model/{subcarpeta}/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [31]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [32]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

## **4. Re-formateo más Encoder mínimo**

### **4.1. Helper: de 2D (aplanado) a 3D (B, T, F)**

Este bloque define una función auxiliar utilizada para **re-formatear las ventanas de datos** desde una representación 2D a la representación 3D requerida por el modelo.

- **Entrada**:
  - `X_flat`: matriz 2D con forma *(N, window_size × n_features)*.
  - `window_size`: longitud temporal de la ventana.
  - `n_features`: cantidad de features por paso temporal.

- **Funcionamiento**:
  - Verifica que la entrada sea efectivamente una matriz 2D.
  - Valida la consistencia dimensional comprobando que  
    `window_size × n_features == X_flat.shape[1]`.
  - Reconvierte los datos al formato *(N, window_size, n_features)* mediante `reshape`.

- **Salida**:
  - Un arreglo 3D listo para ser utilizado como entrada del modelo.

Este helper se utiliza para **cada conjunto de datos (train, valid y test)**.  
En este proyecto se emplea `window_size = 90` y `n_features_90 = 12`, asegurando que cada ventana temporal esté correctamente estructurada antes del entrenamiento.

In [33]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

En este bloque se definen los parámetros estructurales de entrada del modelo:

- `window_size = 90`  
  Establece la longitud de la ventana temporal, es decir, la cantidad de minutos consecutivos utilizados como entrada para cada muestra.

- `features_base`  
  Contiene las variables OHLCV básicas del mercado: *open, high, close, low y volume*.

- `features_90`  
  Se construye combinando las variables base con los factores adicionales definidos en `features_to_90`, conformando el conjunto completo de features utilizadas por el modelo.

- `n_features_90`  
  Representa la cantidad total de features por paso temporal y se obtiene como la longitud de `features_90`.

Este bloque permite **verificar explícitamente** el conjunto de features y su cardinalidad, asegurando coherencia dimensional con la configuración del modelo y las funciones de re-formateo de ventanas.

In [34]:
window_size = 90
features_base = ['open','high','close','low','volume']
features_90 = features_base + features_to_90
n_features_90 = len (features_90)
print(f'features_90:\t\t {features_90}')
print(f'n_features_90:\t {n_features_90}')

features_90:		 ['open', 'high', 'close', 'low', 'volume', 'ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']
n_features_90:	 12


**Re-formateo de ventanas por fold y liberación de memoria**

Este bloque se encarga de **transformar las ventanas de entrada desde formato 2D a formato 3D**, de acuerdo con la configuración definida previamente (`window_size = 90` y `n_features_90 = 12`), y de **optimizar el uso de memoria** durante el procesamiento por fold.

- Para cada *fold* en `k_folds`:
  - Las matrices escaladas `X_train_sc[k]`, `X_valid_sc[k]` y `X_test_sc[k]`, originalmente en formato  
    *(N, window_size × n_features_90)*, se convierten al formato requerido por el modelo:  
    *(N, window_size, n_features_90)* mediante la función `reshape_windows`.
  - Los datos re-formateados se almacenan en los diccionarios `Xtr`, `Xva` y `Xte`, indexados por fold.

- Una vez completado el re-formateo de cada fold:
  - Se eliminan explícitamente las matrices 2D originales para evitar duplicación innecesaria de datos en memoria.
  - Se invoca el recolector de basura (`gc.collect()`) para liberar RAM de forma inmediata.

El objetivo principal es asegurar que cada conjunto de datos esté correctamente estructurado para el entrenamiento del modelo Transformer, manteniendo un consumo de memoria controlado durante el procesamiento de múltiples folds.

In [35]:
import gc

Xtr = {}
Xva = {}
Xte = {}

for k in k_folds:
    Xtr[k] = reshape_windows(X_train_sc[k], window_size, n_features_90)
    Xva[k] = reshape_windows(X_valid_sc[k], window_size, n_features_90)
    Xte[k] = reshape_windows(X_test_sc[k],  window_size, n_features_90)

    # Liberar las matrices 2D de este fold
    del X_train_sc[k], X_valid_sc[k], X_test_sc[k]
    gc.collect()

    print(f'Fold {k} re-shape completo y 2D liberado')

Fold 1 re-shape completo y 2D liberado
Fold 2 re-shape completo y 2D liberado
Fold 3 re-shape completo y 2D liberado
Fold 4 re-shape completo y 2D liberado
Fold 5 re-shape completo y 2D liberado


**Verificación de dimensiones de entrada por fold**

Este bloque define una función auxiliar destinada a **verificar las dimensiones de los datos de entrada** del modelo para cada fold.

- La función itera sobre los *folds* definidos en `k_folds`.
- Para cada fold:
  - Muestra de forma ordenada los *shapes* de los conjuntos **train**, **valid** y **test**.
  - Verifica que cada conjunto se encuentre en formato 3D, consistente con la estructura  
    *(n_samples, window_size, n_features)* requerida por el modelo.

- La salida se presenta en forma tabular, facilitando la inspección visual y la detección temprana de inconsistencias dimensionales entre folds o conjuntos de datos.

El objetivo principal es confirmar que el re-formateo de las ventanas se haya realizado correctamente antes de proceder al entrenamiento y evaluación del modelo.


In [36]:
def mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte):
    for k in k_folds:
        print(f"\nShapes del Fold {k}")
        print(f"{'Set':<10}{'Shape (3D)':<25}")
        print("-" * 40)

        filas = [
            ("Train", Xtr[k].shape),
            ("Valid", Xva[k].shape),
            ("Test",  Xte[k].shape),
        ]

        for nombre, shape_3d in filas:
            print(f"{nombre:<10}{str(shape_3d):<25}")

In [37]:
mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte)


Shapes del Fold 1
Set       Shape (3D)               
----------------------------------------
Train     (124279, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 2
Set       Shape (3D)               
----------------------------------------
Train     (149177, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 3
Set       Shape (3D)               
----------------------------------------
Train     (174075, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 4
Set       Shape (3D)               
----------------------------------------
Train     (198973, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 5
Set       Shape (3D)               
----------------------------------------
Train     (223871, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852

### **4.2. Encoder: backbone + posición + TransformerEncoder**

Este bloque define el **encoder temporal del modelo**, que actúa como *backbone* y es responsable de transformar las ventanas de entrada en **representaciones latentes ricas por paso temporal**, capturando dependencias temporales y relaciones entre features.


#### **4.2.1. Codificación posicional sinusoidal**


La clase `SinusoidalPositionalEncoding` implementa una **codificación posicional determinística**, basada en funciones seno y coseno, tal como fue introducida en el Transformer original.

- Genera una matriz de posiciones de tamaño `(max_len, d_model)` donde:
  - Las posiciones pares usan funciones seno.
  - Las posiciones impares usan funciones coseno.
- Esta codificación:
  - No tiene parámetros entrenables.
  - Permite al modelo incorporar información sobre el **orden temporal** dentro de la ventana.
- Se registra como *buffer* (`register_buffer`), por lo que:
  - Se mueve automáticamente a CPU/GPU.
  - No se actualiza durante el entrenamiento.

En el `forward`, la codificación posicional se **suma** a los embeddings de entrada:
- Entrada: `(B, T, D)`
- Salida: `(B, T, D)`

In [38]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T]

#### **4.2.2. Encoder de series temporales (`TimeSeriesEncoder`)**

La clase `TimeSeriesEncoder` implementa el **encoder completo**, compuesto por tres bloques principales:

1. **Proyección de entrada (backbone lineal)**  
   - Convierte las features originales de cada paso temporal desde:
     ```
     (B, T, F) → (B, T, d_model)
     ```
   - Permite trabajar en un espacio latente de mayor capacidad (`d_model`).

2. **Codificación posicional**  
   - Añade información explícita de posición temporal a cada embedding.
   - Es fundamental para que el Transformer distinga el orden dentro de la ventana.

3. **Transformer Encoder**  
   - Conformado por `num_layers` capas apiladas de `TransformerEncoderLayer`.
   - Cada capa incluye:
     - Multi-Head Self-Attention (`nhead`)
     - Feedforward interno (`dim_feedforward`)
     - Dropout y normalización (*pre-norm*, `norm_first=True`)
   - Opera con `batch_first=True`, manteniendo el formato `(B, T, D)`.

Finalmente, se aplica un `Dropout` adicional como regularización.

---

**Inicialización**

- La capa de proyección (`input_proj`) se inicializa con **Xavier Uniform**, asegurando estabilidad al inicio del entrenamiento.
- El sesgo se inicializa en cero.

---

**Salida del encoder**

El `forward` del encoder devuelve: `(B, T, d_model)`


Es decir, un **embedding contextualizado por cada paso temporal**, que posteriormente será utilizado por el bloque de *pooling* y la cabeza de regresión.


In [39]:
class TimeSeriesEncoder(nn.Module):
    """
    Proyección a d_model + PositionalEncoding + TransformerEncoder (sin cabeza).
    Devuelve embeddings por paso temporal: (B, T, d_model)
    """
    def __init__(
        self,
        input_dim: int,
        d_model: int = 128,
        #Hiperparámetros del modelo
        nhead: int = 8,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

        # init
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        z = self.input_proj(x)    # (B, T, D)
        z = self.pos_encoder(z)   # (B, T, D)
        z = self.encoder(z)       # (B, T, D)
        z = self.dropout(z)       # (B, T, D)
        return z

El objetivo principal es extraer representaciones temporales profundas y contextualizadas de cada ventana intradía, que sirvan como base para la predicción de retornos futuros.

#### **4.2.3. Inicialización de encoders por fold y asignación de dispositivo**


Este bloque se encarga de **instanciar el encoder del modelo para cada fold**, asegurando independencia entre entrenamientos y una correcta gestión del dispositivo de cómputo.

- Se detecta automáticamente el dispositivo disponible:
  - `cuda` si hay GPU disponible.
  - `cpu` en caso contrario.

- Se crea un diccionario `encoders` donde:
  - Cada *fold* posee su **propia instancia** de `TimeSeriesEncoder`.
  - Esto evita compartir pesos entre folds y garantiza aislamiento experimental durante la validación cruzada.

- Para cada fold:
  - Se inicializa el encoder con:
    - `input_dim = n_features_90`, correspondiente al número de features por paso temporal (12).
    - Hiperparámetros arquitectónicos fijados en esta etapa (`d_model`, `nhead`, `num_layers`), los cuales serán candidatos a ajuste en etapas posteriores de *hyperparameter tuning*.
  - El modelo se traslada explícitamente al dispositivo definido (`.to(device)`).

El objetivo principal es disponer de un encoder independiente y correctamente configurado para cada fold, permitiendo entrenamientos reproducibles y comparables dentro del esquema de validación cruzada.


In [40]:
baseline_params = {
    #De arquitectura
    "dropout": 0.1,
    "d_model": 128,
    "n_layers": 2,
    "n_heads": 8,
    "ff_mult": 2,
    "activation": "gelu",
    "pooling": "mean",
    "head_dropout": 0.1,

    #De entrenamiento
    "max_epochs": 50,
    "patience": 8,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
  }


In [41]:
baseline_params

{'dropout': 0.1,
 'd_model': 128,
 'n_layers': 2,
 'n_heads': 8,
 'ff_mult': 2,
 'activation': 'gelu',
 'pooling': 'mean',
 'head_dropout': 0.1,
 'max_epochs': 50,
 'patience': 8,
 'lr': 0.0003,
 'weight_decay': 0.0001,
 'grad_clip': 1.0}

In [42]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Diccionario de encoders por fold
encoders = {}

for k in k_folds:
    encoders[k] = TimeSeriesEncoder(
        input_dim=n_features_90,
        # HIPERPARÁMETROS TUNEADOS Y DIJOS DE heavy_best_params
        d_model = baseline_params['d_model'],
        nhead = baseline_params['n_heads'],
        num_layers = baseline_params['n_layers'],
        dim_feedforward = baseline_params['d_model']*baseline_params['ff_mult'],
        dropout = baseline_params['dropout'],          # default
        activation = baseline_params['activation'],    # default
    ).to(device)
    print(f"Baseline encoder creado para fold {k}")

Baseline encoder creado para fold 1
Baseline encoder creado para fold 2
Baseline encoder creado para fold 3
Baseline encoder creado para fold 4
Baseline encoder creado para fold 5


#### **4.4.4. Verificación del funcionamiento del encoder por fold**

Este bloque define y ejecuta una función de **validación operativa del encoder**, cuyo objetivo es comprobar que el modelo procesa correctamente los datos de entrada y produce salidas con las dimensiones esperadas.

La función `verificar_encoder` recibe como entrada un *fold* específico, los datos re-formateados (`Xtr`, `Xva`, `Xte`) y el diccionario de encoders, y realiza los siguientes pasos:

1. **Selección del dispositivo**  
   Detecta automáticamente si se utilizará GPU (`cuda`) o CPU, garantizando coherencia con el encoder previamente instanciado.

2. **Carga de ventanas 3D**  
   Extrae los conjuntos *train*, *valid* y *test* correspondientes al fold, ya estructurados en formato: `(n_samples, window_size, n_features)`

3. **Creación de mini-batches de inspección**  
    - Selecciona las primeras 64 muestras de cada conjunto.
    - Las convierte a tensores `torch.float32` y las traslada al dispositivo.
    - No se busca entrenar, solo verificar el flujo de datos.

4. **Selección del encoder del fold**  
  Recupera la instancia de `TimeSeriesEncoder` asociada al fold, garantizando consistencia entre datos y modelo.

5. **Ejecución del encoder en modo inferencia**  
    - Ejecuta el encoder dentro de un bloque `torch.no_grad()`.
    - Evita el cálculo de gradientes y reduce el consumo de memoria.
    - Imprime la forma de la salida para cada conjunto.

La salida esperada del encoder es: `(B, T, d_model)`

In [43]:
def verificar_encoder(fold, Xtr, Xva, Xte, encoders):
    print(f"\n=== Fold {fold} ===")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ---------- 1) Cargar ventanas 3D ----------
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    # ---------- 2) Mini-batches para inspección ----------
    xb_tr = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:64], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:64], dtype=torch.float32).to(device)

    # ---------- 3) Encoder del fold ----------
    encoder = encoders[fold]

    # ---------- 4) Pares para inspección ----------
    pairs = [
        (xb_tr, encoder, f"Fold{fold}-train"),
        (xb_va, encoder, f"Fold{fold}-valid"),
        (xb_te, encoder, f"Fold{fold}-test"),
    ]

    # ---------- 5) Ejecutar encoder ----------
    for xb, enc, tag in pairs:
        with torch.no_grad():
            z = enc(xb)
        print(tag, "→", z.shape)



El siguiente bucle aplica esta verificación a todos los folds, confirmando que:

- El encoder acepta correctamente las entradas 3D.
- No existen inconsistencias dimensionales entre folds.
- El backbone del modelo está correctamente configurado antes del entrenamiento.

El objetivo principal es validar de forma temprana la compatibilidad entre los datos y el encoder, evitando errores silenciosos o fallos costosos durante la etapa de entrenamiento y tuneo de hiperparámetros.

In [44]:
for k in k_folds:
  verificar_encoder(k, Xtr, Xva, Xte, encoders)


=== Fold 1 ===
Fold1-train → torch.Size([64, 90, 128])
Fold1-valid → torch.Size([64, 90, 128])
Fold1-test → torch.Size([64, 90, 128])

=== Fold 2 ===
Fold2-train → torch.Size([64, 90, 128])
Fold2-valid → torch.Size([64, 90, 128])
Fold2-test → torch.Size([64, 90, 128])

=== Fold 3 ===
Fold3-train → torch.Size([64, 90, 128])
Fold3-valid → torch.Size([64, 90, 128])
Fold3-test → torch.Size([64, 90, 128])

=== Fold 4 ===
Fold4-train → torch.Size([64, 90, 128])
Fold4-valid → torch.Size([64, 90, 128])
Fold4-test → torch.Size([64, 90, 128])

=== Fold 5 ===
Fold5-train → torch.Size([64, 90, 128])
Fold5-valid → torch.Size([64, 90, 128])
Fold5-test → torch.Size([64, 90, 128])


Los resultados confirman que el **encoder funciona correctamente en todos los folds**.

Para los conjuntos de *train*, *valid* y *test*, el modelo recibe ventanas de longitud **90** y produce embeddings de dimensión **128** por paso temporal, manteniendo de forma consistente la estructura esperada `(batch, window_size, d_model)`.

Esto valida que:
- El re-formateo de las ventanas es correcto.
- La proyección de features y la codificación posicional están correctamente configuradas.
- El *backbone* del Transformer está listo para el entrenamiento y el posterior tuneo de hiperparámetros.


## **5. Pooling (Sin cambiar enconder)**

En esta etapa se introduce el **mecanismo de pooling temporal**, cuyo objetivo es **colapsar la dimensión temporal** de la salida del encoder sin alterar su arquitectura ni sus pesos.


- El encoder produce una salida de forma `(B, T, d_model)` donde cada paso temporal contiene un embedding contextualizado.

- El pooling se aplica **posteriormente al encoder**, transformando la salida a: `(B, d_model)` obteniendo una representación fija por ventana.

Este diseño permite:
- Mantener el encoder como *backbone* reutilizable.
- Evaluar distintos esquemas de agregación temporal sin reentrenar ni redefinir el encoder.
- Comparar el impacto del pooling en el desempeño predictivo del modelo.

- Los métodos de pooling típicos (simples y que no requieren modificar el encoder) incluyen:
  - **Mean pooling**: promedio sobre la dimensión temporal.
  - **Last pooling**: selección del último paso temporal.
  - **Attention pooling**: combinación ponderada aprendida de los pasos temporales.

El objetivo principal es extraer una representación global de cada ventana temporal a partir de los embeddings del encoder, preparándola para su uso en la cabeza de regresión (regresion head) del modelo.

#### **5.1. Pooling Temporal**

Este bloque define un módulo de **pooling temporal**, encargado de **agregar la dimensión temporal** de la salida del encoder sin modificar su estructura ni sus parámetros.

- **Entrada**:
  - Tensor `z` con forma `(B, T, D)`, correspondiente a los embeddings temporales generados por el encoder.

- **Modos soportados**:
  - **`mean`**: realiza el promedio sobre la dimensión temporal, produciendo una representación global que resume toda la ventana.
  - **`last`**: selecciona el embedding del último paso temporal, representando el estado final de la secuencia.

- **Salida**:
  - Tensor con forma `(B, D)`, adecuado para ser utilizado por la cabeza de regresión del modelo.

Este enfoque permite **comparar distintas estrategias de agregación temporal** manteniendo fijo el encoder, facilitando el análisis del impacto del pooling en el desempeño predictivo.

In [45]:
import torch
import torch.nn as nn

class TemporalPooling(nn.Module):
    def __init__(self, mode: str = "mean"):
        super().__init__()
        assert mode in ("mean", "last"), "Soportado: 'mean' o 'last'"
        self.mode = mode

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B, T, D)
        if self.mode == "mean":
            return z.mean(dim=1)      # (B, D)
        else:  # "last"
            return z[:, -1, :]        # (B, D)

####**5.2. Verificación del pooling temporal por fold**

Este bloque se utiliza para **validar el funcionamiento de los mecanismos de pooling temporal**, aplicados sobre la salida del encoder, sin modificar su arquitectura.

- Se instancian dos estrategias de pooling:
  - `pool_mean`: promedio sobre la dimensión temporal.
  - `pool_last`: selección del último paso temporal.

- Para cada *fold*:
  1. Se cargan las ventanas 3D de *train*, *valid* y *test* desde los diccionarios correspondientes.
  2. Se construyen mini-batches de 64 muestras y se trasladan al dispositivo de cómputo disponible.
  3. Se recupera el encoder asociado al fold.
  4. Cada mini-batch se procesa en modo inferencia (`torch.no_grad()`):
     - Primero por el encoder, obteniendo embeddings con forma `(B, T, d_model)`.
     - Luego por ambos métodos de pooling, colapsando la dimensión temporal a `(B, d_model)`.

- Finalmente, se imprimen las dimensiones resultantes para cada conjunto y método de pooling, permitiendo verificar:
  - La coherencia dimensional de las salidas.
  - La correcta integración del pooling con el encoder.

El objetivo principal es confirmar que las estrategias de pooling temporal generan representaciones globales válidas y consistentes, listas para ser utilizadas por la cabeza de regresión del modelo.

In [46]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:
    print(f"\n=== Fold {fold} ===")

    # 1) Cargar ventanas 3D del fold desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    # 2) Mini-batches
    xb_tr = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:64], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold desde diccionario
    enc = encoders[fold]

    # 4) Lista de sets de este fold
    pairs = [
        (xb_tr, enc, f"Fold{fold}-train"),
        (xb_va, enc, f"Fold{fold}-valid"),
        (xb_te, enc, f"Fold{fold}-test"),
    ]

    # 5) Pasar por encoder + pooling
    for xb, encoder, tag in pairs:
        with torch.no_grad():
            z  = encoder(xb)    # (64, T, d_model)
            p1 = pool_mean(z)   # (64, d_model)
            p2 = pool_last(z)   # (64, d_model)
        print(f'Para {tag}\n\tPool mean:\t{p1.shape}\tPool last:\t{p2.shape}')



=== Fold 1 ===
Para Fold1-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 2 ===
Para Fold2-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 3 ===
Para Fold3-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 4 ===
Para Fold4-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-test

#####**5.2.1. Conclusión e interpretación de resultados**

Los resultados obtenidos son **correctos y coherentes** con la arquitectura definida.

- En cada evaluación se utilizan **mini-batches de 64 ventanas** provenientes de los conjuntos *train*, *valid* y *test* de cada fold, por lo que el tamaño de batch es 64.
- El encoder transforma cada entrada en una secuencia de embeddings con forma: `(64, T, 128)` donde `T = 90` corresponde a la longitud de la ventana y `128` es la dimensión latente definida por `d_model`.


- A continuación, los métodos de pooling temporal:
    - `pool_mean(z)` colapsa la dimensión temporal mediante un **promedio**, produciendo un tensor de forma `(64, 128)`.
    - `pool_last(z)` selecciona el **último paso temporal**, generando igualmente un tensor `(64, 128)`.

- La consistencia de las dimensiones entre folds es esperable, ya que:
    - Todos los folds comparten la misma arquitectura (`d_model = 128`).
    - Se utiliza el mismo tamaño de batch para la verificación.

En conjunto, se ha verificado que:

  - Los datos re-formateados (`Xtr_k`, `Xva_k`, `Xte_k`) poseen la forma adecuada y pueden ingresar correctamente al encoder.
  - Los encoders asociados a cada fold están correctamente inicializados y operan sin errores.
  - El módulo `TemporalPooling` funciona según lo esperado y produce **embeddings 2D** con forma `(batch, d_model)`, listos para ser utilizados por la cabeza final del modelo (regresión o clasificación).

Estos resultados confirman que el pipeline **encoder + pooling** está correctamente integrado y preparado para avanzar hacia la definición y entrenamiento de la cabeza de salida (head regression).


### **5.3. Inspección numérica de embeddings tras el pooling temporal**

Este bloque de código se utiliza para **inspeccionar los valores numéricos reales** de los embeddings generados por el modelo, luego de aplicar el encoder y el pooling temporal.

- Se instancian los módulos de pooling (`mean` y `last`), aunque en este caso se utiliza únicamente `mean`.
- Se selecciona automáticamente el dispositivo de cómputo disponible (GPU o CPU).

- Para cada *fold*:
  1. Se cargan las ventanas 3D del conjunto de entrenamiento desde el diccionario `Xtr`.
  2. Se construye un mini-batch de 64 muestras y se convierte a tensor en el dispositivo.
  3. Se recupera el encoder correspondiente al fold.
  4. En modo inferencia (`torch.no_grad()`), se ejecuta:
     - El encoder, obteniendo embeddings temporales con forma `(64, T, 128)`.
     - El pooling medio, colapsando la dimensión temporal a `(64, 128)`.

- Finalmente, se imprimen los **primeros 10 valores numéricos** del embedding correspondiente a la **primera muestra del batch**.

El objetivo principal es verificar que el pipeline *encoder + pooling* no solo produce las dimensiones correctas, sino también **valores numéricos válidos y finitos**, confirmando que los embeddings contienen información continua utilizable por la cabeza de salida del modelo.

In [47]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:

    print(f"\n=== Fold {fold} ===")

    # 1) Cargar ventanas 3D desde diccionario
    Xtr_k = Xtr[fold]

    # 2) Mini-batch
    xb = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold desde diccionario
    enc = encoders[fold]

    # 4) Ejecutar encoder + pooling
    with torch.no_grad():
        z = enc(xb)               # (64, T, 128)
        p_mean = pool_mean(z)     # (64, 128)

    # 5) Mostrar valores numéricos reales del embedding
    print("Primeros 10 valores del embedding del fold:")
    print(p_mean[0, :10].cpu().numpy())



=== Fold 1 ===
Primeros 10 valores del embedding del fold:
[-0.23314852  0.6029641   0.8588653  -0.42319006  0.7064066   0.31639004
  0.18698023  0.16949797  0.31119594  1.8115602 ]

=== Fold 2 ===
Primeros 10 valores del embedding del fold:
[ 0.22785819 -1.1399345  -0.3087484  -0.1306986   0.24426325  0.00922105
 -0.4233306  -0.8209444  -0.23319957 -0.38986343]

=== Fold 3 ===
Primeros 10 valores del embedding del fold:
[ 1.0932766  -0.22211713 -0.2400513   1.1905823  -0.698006    0.4875757
 -0.05589941  0.66490406 -0.35018474  1.6504517 ]

=== Fold 4 ===
Primeros 10 valores del embedding del fold:
[-0.6228314  -0.04714086  0.07346641  1.3440956  -0.39805272  1.0747284
  1.2393951  -2.2453563  -1.1839511  -0.12597795]

=== Fold 5 ===
Primeros 10 valores del embedding del fold:
[-0.31775224  0.68098634 -0.44104043 -0.6663667   0.14795224  0.90852076
 -0.7543027   0.9950121   0.8219513  -0.5834101 ]


#### **5.3.1. Conclusión de la inspección numérica de embeddings**

Los valores impresos para cada fold son **coherentes y esperables**.

- Se observan valores **reales y finitos** (no aparecen `NaN` ni `Inf`), lo que indica que el pipeline: `X (ventanas 3D) → encoder → pooling(mean) → embedding (128)` está funcionando correctamente a nivel numérico.

- Los embeddings presentan valores tanto positivos como negativos, con magnitudes moderadas (aprox. entre -2 y 2 en los ejemplos), lo cual es típico en representaciones latentes generadas por redes neuronales con normalización y dropout.

- Es normal que los primeros 10 valores varíen entre folds, ya que:
  - Cada fold utiliza datos distintos.
  - Cada encoder es una instancia independiente (pesos inicializados separadamente, salvo que se haya fijado una semilla global y un orden de ejecución idéntico).

Esta prueba confirma que el modelo no solo respeta las dimensiones esperadas, sino que también produce embeddings numéricamente válidos, listos para ser consumidos por la cabeza de regresión en las siguientes etapas.

### **5.4. Para observar el contenido de Train, Valid y Test separados**

Este bloque de código se utiliza para **observar y comparar el contenido numérico de los embeddings** generados a partir de los conjuntos *train*, *valid* y *test*, de manera separada, para cada fold.

- Se instancian los módulos de pooling temporal (`mean` y `last`), utilizándose en este caso únicamente el **mean pooling**.
- Se selecciona automáticamente el dispositivo de cómputo disponible (GPU o CPU).

- Para cada *fold*:
  - Se cargan las ventanas 3D correspondientes a los conjuntos *train*, *valid* y *test* desde los diccionarios `Xtr`, `Xva` y `Xte`.
  - Se recupera el encoder asociado a ese fold, garantizando coherencia entre datos y modelo.

- Para cada conjunto (*train*, *valid*, *test*):
  - Se toma un mini-batch de 64 muestras.
  - Se ejecuta el pipeline **encoder + pooling** en modo inferencia (`torch.no_grad()`).
  - Se imprimen los **primeros 10 valores del embedding** correspondiente a la primera muestra del batch.

El objetivo principal es permitir una inspección cualitativa de los embeddings generados a partir de cada conjunto de datos, verificando que:
- El encoder y el pooling producen valores numéricos válidos en todos los sets.
- No existen anomalías evidentes entre *train*, *valid* y *test* antes de avanzar al entrenamiento completo del modelo.

In [48]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:

    print(f"\n=== FOLD {fold} ===")

    # Cargar ventanas desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    sets = {
        "train": Xtr_k,
        "valid": Xva_k,
        "test":  Xte_k
    }

    # Encoder desde diccionario
    enc = encoders[fold]

    for name, X in sets.items():

        xb = torch.tensor(X[:64], dtype=torch.float32).to(device)

        with torch.no_grad():
            z = enc(xb)
            p_mean = pool_mean(z)

        print(f"\n{name.upper()} — primeros 10 valores:")
        print(p_mean[0, :10].cpu().numpy())



=== FOLD 1 ===

TRAIN — primeros 10 valores:
[-0.21976836  0.61161774  0.84065247 -0.43184206  0.65523887  0.38689965
  0.08878414  0.08686467  0.29075798  1.8707404 ]

VALID — primeros 10 valores:
[-0.43419367  0.8085849  -0.7311975  -0.07921455  0.11939153 -1.1788188
  0.13318533 -0.43337148  0.87846994 -0.40570176]

TEST — primeros 10 valores:
[-0.5714438  -0.8529299  -1.1501613   0.4947381  -1.2607809  -0.8422855
  1.6239984  -0.44061247  0.06743129 -2.0269296 ]

=== FOLD 2 ===

TRAIN — primeros 10 valores:
[ 0.21310252 -1.0506523  -0.38603187 -0.0969826   0.20122741 -0.0462056
 -0.4301386  -0.87006974 -0.25384793 -0.39013496]

VALID — primeros 10 valores:
[ 0.31082746 -0.8701214  -0.7608804   0.00534241 -0.5050274   0.3820109
  0.25640061 -0.15281881  0.05734891 -0.04172057]

TEST — primeros 10 valores:
[ 2.2003195  -0.24763162  0.44304937  0.35058594 -1.0431023  -0.1594866
  1.6289712  -0.2607989   0.8561149   0.8490468 ]

=== FOLD 3 ===

TRAIN — primeros 10 valores:
[ 0.9463948

####**5.4.1. Revisión del análisis y conclusión**

Los resultados confirman que el pipeline **fold → encoder → pooling** funciona correctamente. Además:

- Los embeddings difieren entre *train*, *valid* y *test* dentro de cada fold, y también entre folds, lo cual es esperable debido a la segmentación temporal y a la independencia de los encoders.

- No se observan colapsos, valores constantes ni anomalías numéricas (`NaN`/`Inf`).  
- El pooling genera representaciones 2D coherentes, listas para ser utilizadas por la cabeza de salida.

En síntesis, la generación de embeddings es estable y el flujo de datos por fold está correctamente implementado para avanzar al entrenamiento y tuneo del modelo.

##**6. Cabeza de regresión ('Regression Head') - Salida escalar**

La **cabeza de regresión** es el último bloque del modelo y cumple la función de **convertir el embedding generado por el encoder (y el pooling)** en una **predicción continua escalar**.


Conceptualmente:

- Recibe como entrada un embedding con forma: `(B, D)`
- Produce como salida un único valor por muestra: `(B,)`

Ese valor escalar puede representar, según el objetivo definido:
- El **retorno futuro**.
- La **dirección del precio** (si se modela como regresión continua).
- La **magnitud del movimiento**.
- Cualquier otra variable continua de interés.

### **6.1. Implementación de la cabeza de regresión**

La clase `RegressionHead` implementa una **red neuronal totalmente conectada (MLP)** simple y estable, diseñada para transformar el embedding en una predicción final.


**Arquitectura:**

- **Entrada**: embedding de dimensión `d_model` (por ejemplo, 128).
- **Primera capa lineal**: Reduce la dimensión del embedding de `D → D/2`.
- **Activación GELU**: Activación suave y estándar en arquitecturas Transformer.
- **Dropout**: Regularización para reducir el riesgo de *overfitting*.
- **Segunda capa lineal**: Proyecta de `D/2 → 1`, produciendo la salida escalar.

**Flujo dimensional:**

  `(B, D) → (B, D/2) → (B, 1) → (B,)`

  En el método `forward`, se aplica `squeeze(-1)` para eliminar la dimensión final innecesaria y devolver directamente un tensor 1D por batch.

El objetivo principal es transformar la representación latente aprendida por el encoder en una **predicción numérica directa**, cerrando el pipeline completo: `Ventanas → Encoder → Pooling → Regression Head → Predicción`.

Este diseño mantiene la arquitectura modular, permitiendo reutilizar el mismo encoder y pooling con distintas cabezas de salida si se desea cambiar el objetivo del modelo.



In [49]:
baseline_params

{'dropout': 0.1,
 'd_model': 128,
 'n_layers': 2,
 'n_heads': 8,
 'ff_mult': 2,
 'activation': 'gelu',
 'pooling': 'mean',
 'head_dropout': 0.1,
 'max_epochs': 50,
 'patience': 8,
 'lr': 0.0003,
 'weight_decay': 0.0001,
 'grad_clip': 1.0}

In [50]:
class RegressionHead(nn.Module):
    """
    Cabeza de regresión para modelos de series temporales.
    Toma un embedding de dimensión D (por ejemplo, 128) y produce
    un único valor escalar por muestra (predicción continua).
    """

    def __init__(self, d_model: int = 128, dropout: float = 0.1, activation: str = "relu"):
        super().__init__()

        act = self._get_activation(activation)

        # Red neuronal totalmente conectada (MLP) en dos capas:
        # 1) Proyección D -> D/2 con activación GELU.
        # 2) Proyección D/2 -> 1 (salida escalar).
        self.net = nn.Sequential(

            # Primera capa lineal: reduce la dimensión del embedding.
            # Entrada: (B, d_model)
            # Salida:  (B, d_model // 2)
            nn.Linear(d_model, d_model // 2),

            # GELU: activación usada en Transformers, suave y estable.
            nn.GELU(),

            # Dropout: regularización para evitar overfitting
            nn.Dropout(dropout),

            # Segunda capa lineal: produce un solo valor por muestra.
            # Entrada: (B, d_model // 2)
            # Salida:  (B, 1)
            nn.Linear(d_model // 2, 1)
        )

    @staticmethod
    def _get_activation(name: str) -> nn.Module:
        name = name.lower()
        if name == "relu":
            return nn.ReLU()
        if name == "gelu":
            return nn.GELU()
        if name == "tanh":
            return nn.Tanh()
        if name == "silu" or name == "swish":
            return nn.SiLU()
        raise ValueError(f"activation inválida: {name}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass de la cabeza de regresión.

        Parámetros
        ----------
        x : Tensor con forma (B, D)
            D es la dimensión del embedding producido por el encoder.

        Retorna
        -------
        Tensor con forma (B,)
            Un valor escalar predicho por cada muestra del batch.
        """

        # La red produce un tensor de forma (B, 1).
        # squeeze(-1) elimina la última dimensión para dejarlo en (B,).
        return self.net(x).squeeze(-1)

#### **6.1.1. Validación de la `RegressionHead` para la tarea MNQ**


La cabeza de regresión implementada es adecuada para nuestro problema, porque:

- El objetivo del modelo es un **valor escalar continuo**: el **retorno futuro a 90 minutos**.
- Luego del encoder + pooling obtenemos un embedding por muestra con forma: `(B, d_model) = (batch, 128)`
- Necesitamos una función que transforme ese embedding en una **predicción escalar**: `(B, 128) → (B,)`

La arquitectura propuesta (MLP simple con `Linear → GELU → Dropout → Linear`) es un enfoque **estándar y efectivo** para convertir embeddings de Transformers en salidas de regresión en tareas de forecasting, manteniendo buena capacidad de modelado y regularización.


### **6.2. Creación de la cabeza de regresión por fold**

En este bloque se instancian las **cabezas de regresión de forma independiente para cada fold**, manteniendo coherencia con el esquema de validación temporal.

- Se detecta automáticamente el dispositivo de cómputo disponible (GPU o CPU).
- Se crea un diccionario `heads` donde cada fold posee su **propia instancia** de `RegressionHead`.

Para cada fold:
- La cabeza se inicializa con:
  - `d_model = 128`, consistente con la dimensión del embedding producido por el encoder.
  - `dropout = 0.1`, como regularización.
- La instancia se traslada explícitamente al dispositivo definido.

El objetivo principal es asegurar que cada fold cuente con una cabeza de regresión independiente, evitando el uso compartido de parámetros entre folds y garantizando un entrenamiento y evaluación correctamente aislados.

In [51]:
from pandas.core.arrays import base
device = "cuda" if torch.cuda.is_available() else "cpu"

heads = {}   # Diccionario de heads por fold

for k in k_folds:
    heads[k] = RegressionHead(
        d_model=baseline_params['d_model'], #Hiperparametro tuneado
        dropout=baseline_params['head_dropout'],
        activation=baseline_params["activation"]#Hiperparametro tuneado
    ).to(device)

    print(f"Head creado para fold {k}")

Head creado para fold 1
Head creado para fold 2
Head creado para fold 3
Head creado para fold 4
Head creado para fold 5


### **6.3. Sanity check end-to-end (sin entrenar, solo shapes y un MSE “dummy”)**

Con el sanity check vamos a probar rápidamente que todo el pipeline funciona de punta a punta antes de entrenar.

El pipeline completo es:

`ventanas → encoder → pooling → cabeza de regresión → predicción escalar`

Se verifica que:
- Las ventanas ingresen al encoder y produzcan `z` con forma `(B, T, 128)`.
- El pooling reduzca la dimensión temporal y produzca `h` con forma `(B, 128)`.
- La cabeza de regresión entregue `yhat` con forma `(B,)`.
- No existan errores de shapes ni inconsistencias de batch entre las salidas.

Este chequeo **no entrena**, solo confirma que la arquitectura y los componentes por fold están integrados correctamente.


#### **6.3.1. Código de Sanity Check**

La función `sanity_check_pipeline(...)` recorre cada fold y, para cada conjunto (*train*, *valid*, *test*):

1. Recupera desde diccionarios:
   - `Xtr_k`, `Xva_k`, `Xte_k`
   - `encoders[fold]` y `heads[fold]`

2. Toma un mini-batch de tamaño `batch_size` (por defecto 64).

3. Ejecuta el flujo completo (en modo inferencia, sin gradientes):
   - `z = enc(xb)`  
   - `h = pool(z)`  
   - `yhat = head(h)`

4. Comprueba que las formas sean las esperadas:
   - `z` sea 3D
   - `h` sea 2D
   - `yhat` sea 1D
   - El tamaño de batch sea consistente entre `xb`, `h` y `yhat`

5. Imprime por set:
   - Las shapes si todo está correcto, o un mensaje de error si no lo está.
   - Un resumen por fold indicando si el pipeline completo está OK o si hubo problemas.

El objetivo principal es confirmar que el modelo está bien cableado de punta a punta en cada fold, antes de avanzar al entrenamiento y al tuneo de hiperparámetros.

In [52]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Pooling temporal
pool = TemporalPooling(baseline_params['pooling']).to(device)

def sanity_check_pipeline(k_folds, Xtr, Xva, Xte, encoders, heads, batch_size=64):
    """
    Verifica el pipeline completo encoder → pooling → head de regresión
    para cada fold, usando mini-batches chicos.
    """

    for fold in k_folds:
        print(f"\n=== Sanity check FOLD {fold} ===")

        # 1) Recuperar estructuras desde diccionarios
        try:
            Xtr_k = Xtr[fold]
            Xva_k = Xva[fold]
            Xte_k = Xte[fold]

            enc  = encoders[fold]
            head = heads[fold]

        except KeyError as e:
            print(f"  Faltan datos o modelos para el fold {fold}: {e}")
            continue

        sets = {
            "train": Xtr_k,
            "valid": Xva_k,
            "test":  Xte_k
        }

        fold_ok = True

        for nombre_set, X in sets.items():

            if X is None or len(X) == 0:
                print(f"  {nombre_set}: sin datos, se omite.")
                continue

            # 2) Mini-batch chico
            xb = torch.tensor(X[:batch_size], dtype=torch.float32).to(device)

            with torch.no_grad():
                # Paso 1: encoder
                z = enc(xb)          # (B, T, D)

                # Paso 2: pooling
                h = pool(z)          # (B, D)

                # Paso 3: head de regresión
                yhat = head(h)       # (B,)

            # 3) Comprobación de shapes
            ok_shapes = (
                z.ndim == 3 and
                h.ndim == 2 and
                yhat.ndim == 1 and
                xb.shape[0] == h.shape[0] == yhat.shape[0]
            )

            if ok_shapes:
                print(f"  {nombre_set}: z{tuple(z.shape)} → h{tuple(h.shape)} → yhat{tuple(yhat.shape)}")
            else:
                print(f"  {nombre_set}: SHAPES ERROR: "
                      f"z{tuple(z.shape)}, h{tuple(h.shape)}, yhat{tuple(yhat.shape)}")
                fold_ok = False

        if fold_ok:
            print(f"  ✔️ Pipeline COMPLETO OK en FOLD {fold}.")
        else:
            print(f"  ❌ Problemas de shapes en FOLD {fold}.")


In [53]:
sanity_check_pipeline(k_folds, Xtr, Xva, Xte, encoders, heads)


=== Sanity check FOLD 1 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 1.

=== Sanity check FOLD 2 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 2.

=== Sanity check FOLD 3 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 3.

=== Sanity check FOLD 4 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 4.

=== Sanity check FOLD 5 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → 

#### **6.3.2. Conclusión del sanity check end-to-end**

Los resultados confirman que el **pipeline completo del modelo funciona correctamente en todos los folds**.

Para cada fold y para los conjuntos *train*, *valid* y *test* se verifica que:
- El encoder produce salidas con forma `(64, 90, 128)`.
- El pooling reduce correctamente la dimensión temporal a `(64, 128)`.
- La cabeza de regresión genera predicciones escalares con forma `(64,)`.

No se detectan errores de *shape*, inconsistencias de batch ni problemas de integración entre módulos.

En síntesis, la arquitectura **ventanas → encoder → pooling → RegressionHead → predicción escalar** está correctamente conectada y validada, y el modelo se encuentra listo para iniciar la etapa de entrenamiento y tuneo de hiperparámetros.

Antes de entrenar, es fundamental asegurarnos de que:

- Las ventanas están bien formateadas (3D correctas).  
- El encoder procesa correctamente la secuencia.  
- El pooling reduce correctamente la dimensión temporal.  
- La cabeza de regresión produce un escalar por muestra.  
- Todo funciona en CPU o GPU sin errores.

Este paso nos garantiza que el pipeline entero está sano y listo para el entrenamiento real.

### **6.4. Sanity check de pérdida (MSE).**

Este bloque de código se utiliza para **generar predicciones escalares (`ŷ`)** a partir del pipeline completo del modelo, **sin realizar entrenamiento**, únicamente para verificar el flujo end-to-end y preparar estructuras de salida.

- Se define el dispositivo de cómputo (CPU o GPU) y un `batch_size` fijo de 64.
- Se instancia el módulo de **pooling temporal** (`mean`) en el dispositivo.
- Se inicializan diccionarios para almacenar las predicciones de *train*, *valid* y *test* por fold.




#### **6.4.1. Generación de ŷ por fold (inferencia sin entrenamiento)**



Para cada *fold*:

1. **Carga de datos y modelos**  
   - Se recuperan las ventanas 3D (`Xtr_k`, `Xva_k`, `Xte_k`).
   - Se obtienen el `encoder` y la `RegressionHead` correspondientes al fold.

2. **Selección de mini-batches**  
   - Se toman las primeras `batch_size` muestras de cada conjunto.
   - Se convierten a tensores y se envían al dispositivo.

3. **Ejecución del pipeline completo (sin gradientes)**  
   Para *train*, *valid* y *test*: `ventanas → encoder → pooling → RegressionHead → ŷ`

    - El encoder produce `(B, T, D)`.
    - El pooling reduce a `(B, D)`.
    - La cabeza de regresión genera predicciones `(B,)`.

4. **Almacenamiento de predicciones**  
    - Las predicciones se pasan a CPU y se guardan como `numpy arrays` en diccionarios:
      - `yhat_train[fold]`
      - `yhat_valid[fold]`
      - `yhat_test[fold]`

5. **Verificación de shapes**  
    - Se imprimen las dimensiones de cada `ŷ` para confirmar coherencia.

El objetivo principal es:

- Confirmar que el modelo produce **predicciones escalares válidas** para cada conjunto y fold.
- Verificar que el pipeline completo funciona sin errores numéricos o de *shape*.
- Preparar estructuras de salida (`ŷ`) que luego podrán compararse con los valores reales (`y`) o utilizarse en métricas, todo **antes del entrenamiento real**.

In [54]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

# Pooling temporal
pool = TemporalPooling(baseline_params['pooling']).to(device)

# Diccionarios para guardar las predicciones
yhat_train = {}
yhat_valid = {}
yhat_test  = {}

for fold in k_folds:
    print(f"\n=== Generando yhat para Fold {fold} ===")

    # 1) Datos del fold desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    enc  = encoders[fold].to(device)
    head = heads[fold].to(device)

    # 2) Mini‐batches (solo las primeras batch_size muestras)
    xb_tr = torch.tensor(Xtr_k[:batch_size], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:batch_size], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:batch_size], dtype=torch.float32).to(device)

    with torch.no_grad():
        # ---- TRAIN ----
        z_tr    = enc(xb_tr)          # (B, T, D)
        h_tr    = pool(z_tr)          # (B, D)
        yhat_tr = head(h_tr)          # (B,)
        yhat_train[fold] = yhat_tr.cpu().numpy()

        # ---- VALID ----
        z_va    = enc(xb_va)
        h_va    = pool(z_va)
        yhat_va = head(h_va)
        yhat_valid[fold] = yhat_va.cpu().numpy()

        # ---- TEST ----
        z_te    = enc(xb_te)
        h_te    = pool(z_te)
        yhat_te = head(h_te)
        yhat_test[fold] = yhat_te.cpu().numpy()

    print(f"  yhat_train[{fold}].shape =", yhat_train[fold].shape)
    print(f"  yhat_valid[{fold}].shape =", yhat_valid[fold].shape)
    print(f"  yhat_test[{fold}].shape  =", yhat_test[fold].shape)



=== Generando yhat para Fold 1 ===
  yhat_train[1].shape = (64,)
  yhat_valid[1].shape = (64,)
  yhat_test[1].shape  = (64,)

=== Generando yhat para Fold 2 ===
  yhat_train[2].shape = (64,)
  yhat_valid[2].shape = (64,)
  yhat_test[2].shape  = (64,)

=== Generando yhat para Fold 3 ===
  yhat_train[3].shape = (64,)
  yhat_valid[3].shape = (64,)
  yhat_test[3].shape  = (64,)

=== Generando yhat para Fold 4 ===
  yhat_train[4].shape = (64,)
  yhat_valid[4].shape = (64,)
  yhat_test[4].shape  = (64,)

=== Generando yhat para Fold 5 ===
  yhat_train[5].shape = (64,)
  yhat_valid[5].shape = (64,)
  yhat_test[5].shape  = (64,)


#### **6.4.2. Sanity check de MSE por fold (predicciones vs targets escalados)**

Este bloque realiza un **chequeo numérico básico** comparando las predicciones generadas (`ŷ`) con los valores reales escalados (`y`) para cada fold, sin entrenar el modelo.

Para cada fold:

1. **Carga los targets reales** (*train*, *valid* y *test*) desde:
   - `y_train_sc`, `y_valid_sc`, `y_test_sc`.

2. **Recupera las predicciones previamente generadas**:
   - `yhat_train`, `yhat_valid`, `yhat_test`.

3. **Selecciona un mini-batch** de tamaño `batch_size` (por defecto 64) para cada set.

4. **Convierte a tensores** las predicciones y los valores reales, asegurando:
   - Tipo `float32`.
   - Forma 1D `(B,)`.
   - Dispositivo consistente (CPU/GPU).

5. **Calcula el error cuadrático medio (MSE)** entre `ŷ` y `y`: `MSE(ŷ, y)`

6. **Imprime por set**:
    - La forma de las predicciones.
    - El valor numérico del MSE.


El objetivo principal es verificar que:
- Las predicciones y los targets tienen **formas compatibles**.
- El cálculo de la función de pérdida funciona correctamente.
- No existen errores numéricos (`NaN`, `Inf`) ni inconsistencias de dispositivo.

Este sanity check **no evalúa performance**, solo confirma que el modelo puede calcular una pérdida válida antes de iniciar el entrenamiento real.


In [55]:
def sanity_check_mse_folds(k_folds, y_train_sc, y_valid_sc, y_test_sc,
                           yhat_train, yhat_valid, yhat_test,
                           batch_size=64):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    for fold in k_folds:
        print(f"\n=== Sanity check MSE — Fold {fold} ===")

        # 1) Targets del fold desde diccionarios
        ytr = y_train_sc[fold]
        yva = y_valid_sc[fold]
        yte = y_test_sc[fold]

        # 2) Predicciones generadas anteriormente
        yhat_tr = yhat_train[fold]
        yhat_va = yhat_valid[fold]
        yhat_te = yhat_test[fold]

        sets = [
            ("train", ytr[:batch_size], yhat_tr[:batch_size]),
            ("valid", yva[:batch_size], yhat_va[:batch_size]),
            ("test",  yte[:batch_size], yhat_te[:batch_size]),
        ]

        for name, y_true, y_pred in sets:

            # Convertir a tensores
            yb = torch.tensor(y_true, dtype=torch.float32, device=device).view(-1)
            yh = torch.tensor(y_pred, dtype=torch.float32, device=device).view(-1)

            # Calcular MSE
            loss = torch.nn.functional.mse_loss(yh, yb)

            print(f"  {name:<6} — yhat:{tuple(yh.shape)}  MSE={float(loss):.6f}")


In [56]:
sanity_check_mse_folds(
    k_folds,
    y_train_sc, y_valid_sc, y_test_sc,
    yhat_train, yhat_valid, yhat_test
)


=== Sanity check MSE — Fold 1 ===
  train  — yhat:(64,)  MSE=0.002870
  valid  — yhat:(64,)  MSE=0.037982
  test   — yhat:(64,)  MSE=0.038646

=== Sanity check MSE — Fold 2 ===
  train  — yhat:(64,)  MSE=0.015933
  valid  — yhat:(64,)  MSE=0.024546
  test   — yhat:(64,)  MSE=0.044726

=== Sanity check MSE — Fold 3 ===
  train  — yhat:(64,)  MSE=0.006405
  valid  — yhat:(64,)  MSE=0.009295
  test   — yhat:(64,)  MSE=0.013583

=== Sanity check MSE — Fold 4 ===
  train  — yhat:(64,)  MSE=0.144851
  valid  — yhat:(64,)  MSE=0.001362
  test   — yhat:(64,)  MSE=0.024304

=== Sanity check MSE — Fold 5 ===
  train  — yhat:(64,)  MSE=0.101898
  valid  — yhat:(64,)  MSE=0.032345
  test   — yhat:(64,)  MSE=0.016130


### **6.4.3. Conclusiones del sanity check de MSE por fold**

A partir de los valores obtenidos de MSE para cada fold, se pueden establecer las siguientes conclusiones:

1. El pipeline funciona correctamente en todos los folds

    - En todos los casos, las predicciones `yhat` presentan la forma esperada `(64,)`.
    - No se registraron errores de dimensiones, tipos de datos ni conflictos de dispositivo (CPU/GPU).
    - Esto confirma que el flujo completo `encoder → pooling → RegressionHead` opera sin inconsistencias técnicas.

2. Los valores de MSE son coherentes con un modelo no entrenado

    - Los pesos del encoder y de la cabeza de regresión **aún no han sido entrenados**, por lo que las predicciones son esencialmente aleatorias.
    - En este contexto:
    - Es esperable que el MSE varíe entre folds y entre conjuntos (*train*, *valid* y *test*).
    - El objetivo de este chequeo no es minimizar el error, sino verificar que el cálculo de la pérdida sea posible y estable.
    - Los valores observados reflejan este comportamiento, con MSE de distinta magnitud según el fold y el conjunto, lo cual es normal en esta etapa.

3. El modelo está listo para avanzar al entrenamiento real

    - El pipeline completo ha sido validado tanto en términos de **formas** como de **cálculo de la función de pérdida**.
    - Se verificó exitosamente que:
    - `ventanas → encoder → pooling → head → yhat`
    - `yhat` puede compararse con los targets reales mediante MSE sin errores.
    - El siguiente paso consiste en implementar el bucle de entrenamiento por fold, incorporando:
    - función de pérdida,
    - optimizador,
    - (opcionalmente) *learning rate scheduling*,
    - métricas de evaluación (RMSE, MAE, SMAPE, Directional Accuracy).

En síntesis, estos resultados confirman que el modelo se encuentra **estructuralmente sano** y preparado para iniciar el entrenamiento sin inconvenientes.

## **7. Preparación para Entrenamiento**

### **7.1. Dataset + DataLoader (reshape dentro)**

El siguiente apartado prepara todo lo necesario para entrenar un modelo en PyTorch usando nuestras ventanas:

1. Escala los valores objetivo (y) usando StandardScaler.
    - Esto ayuda a estabilizar el entrenamiento.
    - El scaler se ajusta solo con y_train (buena práctica).

2. Convierte tus ventanas X (aplanadas en 2D) a tensores 3D (B, T, F)
donde:
    - B = batch size
    - T = tamaño de la ventana temporal (90 minutos)
    - F = número de features

3. Construye un Dataset personalizado (WindowDataset)
    - Guarda X y y en formato listo para PyTorch.
    - Aplica el escalador únicamente a y.

4. Crea dataloaders para entrenamiento y validación
    - `dl_tr`: con shuffle=True
    - `dl_va`: sin shuffle, para evaluación estable
    - Ambos con pin_memory=True (optimiza transferencias CPU→GPU)

Este bloque no entrena nada todavía, pero prepara correctamente los datos para alimentar el modelo fold por fold.

In [57]:
import os, joblib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

'''1) Scaler de y (fit solo con y_train)
 Propósito: Normalizar y para facilitar el entrenamiento y evitar escalas muy pequeñas o muy grandes.
'''
def get_y_scaler(y_train: np.ndarray, path: str = None):
    # Crea un StandardScaler y lo ajusta solo con y_train.
    scaler = StandardScaler()
    scaler.fit(y_train.reshape(-1, 1))   # y debe ser columna

    # Si se pasa un path, guarda el scaler en disco.
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(scaler, path)

    return scaler
'''
2) Dataset que aplica el y_scaler
Propósito: PyTorch necesita un Dataset para entregar lotes de entrenamiento.
Aquí se reconstruyen las ventanas (T,F) y se devuelven como tensores.
'''
class WindowDataset(Dataset):
    def __init__(self, X_flat, y, T, F, y_scaler: StandardScaler):
        # Verifica que X_flat tenga la forma correcta: (N, T*F)
        assert X_flat.shape[1] == T * F, f"Inconsistente: {X_flat.shape[1]} != {T}*{F}"

        # Convierte ventana 2D a 3D: (N, T*F) → (N, T, F)
        X = X_flat.reshape(-1, T, F).astype(np.float32)

        # Si usamos scaler, transformamos y y lo convertimos a float32
        if y_scaler is not None:
            y = y_scaler.transform(y.reshape(-1, 1)).ravel()

        self.X = X
        self.y = y.astype(np.float32)

    def __len__(self):
        # Cantidad total de muestras
        return len(self.y)

    def __getitem__(self, i):
        # Devuelve la i-ésima ventana y su target como tensores PyTorch
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i], dtype=torch.float32)

'''
3) Loaders genéricos (cualquier horizonte)
Propósito: Generar los iteradores que el modelo usará durante el entrenamiento:
      - dl_tr: batches mezclados
      - dl_va: batches ordenados (evaluación estable)
'''

def make_loaders(
    Xtr, ytr, Xva, yva, T, F, y_scaler,
    bs=256,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
):
    ds_tr = WindowDataset(Xtr, ytr, T, F, y_scaler=y_scaler)
    ds_va = WindowDataset(Xva, yva, T, F, y_scaler=y_scaler)

    #persistent_workers solo tiene sentido si num_workers > 0
    persistent_workers = bool(persistent_workers and num_workers > 0)

    dl_tr = DataLoader(
        ds_tr,
        batch_size=bs,
        shuffle=True,
        pin_memory=pin_memory,
        num_workers=num_workers,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor if num_workers > 0 else None,
    )

    dl_va = DataLoader(
        ds_va,
        batch_size=bs,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=num_workers,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor if num_workers > 0 else None,
    )

    return dl_tr, dl_va


El bloque anterior construye el pipeline que convierte tus dataframes en: `Ventanas 3D → Dataset PyTorch → DataLoader → Entrenamiento`

Transforma:
 - X a (B, T, F)
 - y a valores escalados

### **7.2. Modelo compacto por fold (encoder + pooling mean + head)**

Un modelo compacto por fold: `modelo_k = encoder_k + pooling + head_k`

Un modelo compacto por fold combina las tres partes del pipeline (encoder → pooling → head) en un único `nn.Module`.  

Se decidió utilizar un modelo compacto por las siguientes razones:

1. Permite que **cada fold tenga un modelo completamente independiente**, evitando fuga de información entre folds.  
2. Simplifica el loop de entrenamiento: en lugar de ejecutar manualmente `encoder → pool → head`, el modelo produce directamente la predicción `ŷ = model(x)`.  
3. Facilita el uso de optimizadores, carga/guardado de pesos y evaluación, ya que todos los parámetros entrenables quedan dentro de un único módulo por fold.  
4. Mantiene una estructura clara: el “modelo” es la combinación natural de encoder, reducción temporal y cabeza de regresión.

Con esto, el punto 7.3 (loop de entrenamiento) puede trabajar con un único módulo (`model_k`) por fold, lo cual hace el código más limpio y menos propenso a errores.


In [58]:
print(pool)

TemporalPooling()


In [59]:
#assert pool.mode == "last", f"Pooling incorrecto: {pool.mode}"
#print("✔ TemporalPooling configurado en 'last'")

In [60]:
baseline_models = {}   # diccionario para almacenar modelos completos por fold

for fold in k_folds:

    encoder = encoders[fold]
    head    = heads[fold]

    # pooling es compartido
    model = nn.Sequential(
        encoder,   # (B, T, F) → (B, T, d_model)
        pool,      # (B, T, d_model) → (B, d_model)
        head       # (B, d_model) → (B,)
    )

    baseline_models[fold] = model



In [61]:
#Como acceder
#modelo_fold_3 = models[3]
#y_pred = modelo_fold_3(x_batch)

### **7.3. Loop de entrenamiento (MSE, AdamW, early stopping simple)**

El siguiente bloque implementa el loop de entrenamiento del modelo por fold.
Entrena un encoder + pooling + cabeza de regresión usando MSE como función de pérdida, optimizador AdamW, soporte opcional para AMP (mixed precision), clipping de gradiente y un esquema simple de early stopping basado en la pérdida de validación. El objetivo es obtener un modelo estable y con buena generalización, ajustando solo los parámetros del encoder y de la cabeza,mientras que el pooling permanece fijo.


In [62]:
windows_size = 90

In [63]:
# ----------------------------------------------------------------------------------
# NOTA DE DISEÑO
# ----------------------------------------------------------------------------------
# Esta función unifica el entrenamiento del modelo para:
#   1) Entrenamiento baseline
#   2) Coarse / fine tuning con Optuna
#
# La lógica de entrenamiento (forward, backward, AMP, early stopping) es EXACTAMENTE
# la misma en ambos casos, garantizando comparabilidad directa de resultados.
#
# La única diferencia al usar Optuna es:
#   - El reporte de la métrica de validación por época (trial.report)
#   - El pruning temprano de trials poco prometedores (trial.should_prune)
#
# Cuando `trial=None`, la función se comporta como un trainer estándar.
# Cuando `trial` es provisto, se activa automáticamente el modo Optuna.
# ----------------------------------------------------------------------------------

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import optuna

def train_model(
    model: nn.Module,
    dl_tr: DataLoader,
    dl_va: DataLoader,
    device: str = None,

    # -------------------------
    # Hiperparámetros de entrenamiento
    # -------------------------
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    max_epochs: int = 50,
    patience: int = 8,
    grad_clip: float = 1.0,
    use_amp: bool = True,

    # -------------------------
    # Optuna (opcional)
    # -------------------------
    trial: "optuna.trial.Trial" = None,
    prune_after_epoch: int = 4,
):
    """
    Entrena un modelo end-to-end (encoder + pooling + head) usando:
    - Loss: MSE
    - Optimizador: AdamW
    - Early stopping por pérdida de validación
    - Mixed Precision (AMP) opcional
    - Reporting y pruning opcional para Optuna

    La misma función se usa tanto para:
    - Entrenamiento baseline (trial=None)
    - Coarse/Fine tuning con Optuna (trial != None)
    """

    # ==========================================================
    # 0) Selección de dispositivo
    # ==========================================================
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Enviar el modelo completo al dispositivo
    model = model.to(device)

    # ==========================================================
    # 1) Optimizador
    # ==========================================================
    # AdamW es el optimizador recomendado para Transformers
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # ==========================================================
    # 2) Configuración de AMP (solo en GPU)
    # ==========================================================
    amp_enabled = bool(use_amp and device == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    # ==========================================================
    # 3) Variables para early stopping
    # ==========================================================
    best_val = float("inf")   # Mejor pérdida de validación observada
    best_state = None         # Pesos del mejor modelo
    noimp = 0                 # Épocas consecutivas sin mejora

    # ==========================================================
    # 4) Loop principal de entrenamiento
    # ==========================================================
    for epoch in range(1, max_epochs + 1):

        # -------------------------
        # ENTRENAMIENTO
        # -------------------------
        model.train()
        tr_loss = 0.0

        for xb, yb in dl_tr:
            # Enviar batch a GPU/CPU
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            # Reset del gradiente
            opt.zero_grad(set_to_none=True)

            # Forward pass (con AMP si corresponde)
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                yhat = model(xb).view(-1)   # Salida escalar (B,)
                loss = nn.functional.mse_loss(yhat, yb)

            # Backpropagation con escalado de gradiente
            scaler.scale(loss).backward()

            # Necesario antes de aplicar clipping
            scaler.unscale_(opt)

            # Gradient clipping para evitar explosión de gradientes
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            # Paso del optimizador
            scaler.step(opt)
            scaler.update()

            # Acumulación de la pérdida ponderada por tamaño del batch
            tr_loss += loss.item() * xb.size(0)

        # Pérdida media de entrenamiento
        tr_loss /= len(dl_tr.dataset)

        # -------------------------
        # VALIDACIÓN
        # -------------------------
        model.eval()
        va_loss = 0.0

        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=amp_enabled):
                    yhat = model(xb).view(-1)
                    loss = nn.functional.mse_loss(yhat, yb)

                va_loss += loss.item() * xb.size(0)

        # Pérdida media de validación
        va_loss /= len(dl_va.dataset)

        # -------------------------
        # OPTUNA: reporting y pruning (opcional)
        # -------------------------
        if trial is not None and epoch >= prune_after_epoch:
            # Reporta la métrica a Optuna
            trial.report(float(va_loss), step=epoch)

            # Permite a Optuna abortar trials malos temprano
            if trial.should_prune():
                raise optuna.TrialPruned()

        # -------------------------
        # EARLY STOPPING
        # -------------------------
        if va_loss < best_val - 1e-9:
            # Mejora en validación → guardar modelo
            best_val = va_loss
            noimp = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            # No mejora
            noimp += 1
            if noimp >= patience:
                break

    # ==========================================================
    # 5) Restaurar mejores pesos
    # ==========================================================
    if best_state is not None:
        model.load_state_dict(best_state)

    # Se devuelve:
    # - el modelo con los mejores pesos
    # - la mejor pérdida de validación (útil para Optuna)
    return model, float(best_val)

In [64]:
## Para ejecutar el código

'''
for fold in k_folds:
    model_k = globals()[f"model_{fold}"]
    dl_tr_k, dl_va_k = ...  # loaders del fold k
    print(f"\n=== Entrenando modelo del Fold {fold} ===")
    model_k = train_model(model_k, dl_tr_k, dl_va_k)
    globals()[f"model_{fold}"] = model_k
'''

'\nfor fold in k_folds:\n    model_k = globals()[f"model_{fold}"]\n    dl_tr_k, dl_va_k = ...  # loaders del fold k\n    print(f"\n=== Entrenando modelo del Fold {fold} ===")\n    model_k = train_model(model_k, dl_tr_k, dl_va_k)\n    globals()[f"model_{fold}"] = model_k\n'

### **7.4. Inferencia (Predicción)**

La siguiente función realiza inferencia (predicción) en un conjunto completo de ventanas sin calcular gradientes, usando el pipeline: `encoder → pooling → head → predicción escalar`

Sirve para obtener todas las predicciones de train, valid o test después de entrenar el modelo por fold.

En detalle:

1. Convierte X_flat (que viene en formato (N, T*F)) a ventanas 3D (N, T, F)
2. Pasa por el modelo en batches grandes (4096 por defecto) para acelerar la inferencia
3. Obtiene las predicciones ŷ
4. Si las predicciones están escaladas, aplica inverse_transform del scaler de y
5. Devuelve un array 1D con las predicciones reales

Es decir: **Esta función transforma un dataset completo en sus predicciones finales del modelo.**

Se usa después de entrenar, tipicamente para:
  - Evaluar rendimiento
  - Graficar pred vs real
  - Guardar resultados
  - Calcular RMSE, MAE, SMAPE, DA, etc.

In [65]:
@torch.no_grad()
def predict_set(enc, pool, head, X_flat, T, F, device, batch_size=4096, y_scaler=None):
    N = X_flat.shape[0]
    preds = []

    enc.eval(); head.eval()

    for i in range(0, N, batch_size):
        xb_np = X_flat[i:i+batch_size].reshape(-1, T, F).astype(np.float32, copy=False)
        xb = torch.from_numpy(xb_np).to(device, non_blocking=True)

        z = enc(xb)
        h = pool(z)
        yb = head(h).detach().cpu().numpy()
        preds.append(yb)

        # libera referencia del batch (ayuda al GC)
        del xb, z, h

    y_pred_scaled = np.concatenate(preds, axis=0).reshape(-1, 1)
    return (y_scaler.inverse_transform(y_pred_scaled).ravel() if y_scaler is not None else y_pred_scaled.ravel())


En resumen:
- Esta función realiza predicción vectorizada, sin gradientes.
- Usa el pipeline completo: encoder → pooling → head.
- Procesa el dataset en batches grandes (eficiente).
- Reconstruye ventanas desde 2D → 3D.
- Aplica inverse_transform del scaler de y si corresponde.
- Devuelve un vector plano con todas las predicciones del modelo.

# FINE TUNING

## **8. Preparación para fine tuning**

### **8.1. Guardado de información (Optuna + resultados)**

En esta etapa se definen y utilizan rutas de salida para guardar:

**Del estudio de Optuna (una sola vez):**
- `study.pkl`: objeto completo del estudio (permite reanudar o auditar).
- `best_params.json`: mejores hiperparámetros encontrados.
- `trials.csv`: tabla con todos los trials y su score/métricas.

**Resultados finales:**
- `checkpoint_fold_k.pt`: checkpoint por fold (pesos + scaler_y + métricas + hparams).
- `final_metrics.csv`: resumen final (una fila por fold + promedios).
- *(opcional)* `notes.txt`: fecha, dataset tag, features, horizonte, etc.

In [82]:
import os, json
import pandas as pd

# -----------------------------
# 1) Carpeta base (ya la tienes)
# -----------------------------

drive_path_fine_tuning = drive_path + "/5_transformer_model/5_5_fine_tuning"
base_tuned = drive_path_fine_tuning
os.makedirs(base_tuned, exist_ok=True)

# -----------------------------
# 2) Rutas para Optuna (nuevo)
# -----------------------------
optuna_dir = os.path.join(base_tuned, "optuna_study")
os.makedirs(optuna_dir, exist_ok=True)

paths_study = {
    "study_pkl":  os.path.join(optuna_dir, "fine_study.pkl"),
    "best_json":  os.path.join(optuna_dir, "fine_best_params.json"),
    "trials_csv": os.path.join(optuna_dir, "fine_trials.csv"),
}

# -----------------------------
# 3) Rutas para resumen final (nuevo)
# -----------------------------
path_final_metrics = os.path.join(base_tuned, "fine_final_metrics.csv")

paths_study, path_final_metrics

({'study_pkl': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/optuna_study/fine_study.pkl',
  'best_json': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/optuna_study/fine_best_params.json',
  'trials_csv': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/optuna_study/fine_trials.csv'},
 '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_final_metrics.csv')

In [83]:
import json
import joblib
import pandas as pd

def save_fine_tuning_artifacts(
    study,
    metrics_by_fold: dict,
    paths_study: dict,
    path_final_metrics: str,
):
    """
    Guarda todos los artefactos del fine tuning:
      - Estudio completo de Optuna
      - Mejores hiperparámetros
      - Tabla de trials
      - Resumen final de métricas por fold (+ promedio)

    Parámetros
    ----------
    study : optuna.study.Study
        Estudio de Optuna ya finalizado.
    metrics_by_fold : dict
        Métricas por fold devueltas por run_folds_with_hp().
    paths_study : dict
        Diccionario con rutas:
          - study_pkl
          - best_json
          - trials_csv
    path_final_metrics : str
        Ruta donde guardar final_metrics.csv
    """

    # --------------------------------------------------
    # A) Guardar estudio Optuna
    # --------------------------------------------------
    joblib.dump(study, paths_study["study_pkl"])

    with open(paths_study["best_json"], "w") as f:
        json.dump(study.best_params, f, indent=2)

    df_trials = study.trials_dataframe()
    df_trials.to_csv(paths_study["trials_csv"], index=False)

    print("✔ Guardado Optuna:")
    print(" -", paths_study["study_pkl"])
    print(" -", paths_study["best_json"])
    print(" -", paths_study["trials_csv"])

    # --------------------------------------------------
    # B) Guardar resumen final de métricas
    # --------------------------------------------------
    rows = []
    for fold, d in metrics_by_fold.items():
        row = {"fold": fold}
        for split in ["train", "valid", "test"]:
            for mname, mval in d[split].items():
                row[f"{split}_{mname}"] = mval
        rows.append(row)

    df_final = pd.DataFrame(rows).sort_values("fold")

    # Promedio entre folds
    avg = {"fold": "mean"}
    for c in df_final.columns:
        if c != "fold":
            avg[c] = df_final[c].mean()

    df_final = pd.concat(
        [df_final, pd.DataFrame([avg])],
        ignore_index=True
    )

    df_final.to_csv(path_final_metrics, index=False)
    print("✔ Guardado resumen final:", path_final_metrics)

In [84]:
HOW_TO_SAVE = '''
save_fine_tuning_artifacts(
    study=study,
    metrics_by_fold=metrics_by_fold,
    paths_study=paths_study,
    path_final_metrics=path_final_metrics,
)
'''

### **8.2. Carga de los mejores hiperparámetros del Coarse Tuning**

En este paso se cargan desde disco los hiperparámetros óptimos obtenidos durante el proceso de `coarse tuning` con Optuna.  
Esto permite reutilizar los resultados sin necesidad de volver a ejecutar el estudio.

In [85]:
#ruta: '/content/drive/MyDrive/neural_profit/5_transformer_model/5_4_coarse_tuning/optuna_study/coarse_best_params.json'
import json

# Ruta al archivo con los mejores hiperparámetros
path_best_params = (
    "/content/drive/MyDrive/neural_profit/"
    "5_transformer_model/5_4_coarse_tuning/optuna_study/coarse_best_params.json"
)

# Cargar hiperparámetros
with open(path_best_params, "r", encoding="utf-8") as f:
    coarse_best_params = json.load(f)

coarse_best_params = coarse_best_params["best_params"]

In [86]:
coarse_best_params

{'lr': 0.0004524540905246327,
 'weight_decay': 8.724750206557917e-06,
 'grad_clip': 1.6583657701892545,
 'dropout': 0.14334772201167206,
 'd_model': 64,
 'n_layers': 4,
 'n_heads': 8,
 'ff_mult': 4,
 'activation': 'relu',
 'pooling': 'last',
 'head_dropout': 0.11551844963057976}

A partir de este punto, el diccionario `coarse_best_params` se considera **fijo** y se utiliza para:
- Re-instanciar la arquitectura del modelo
- Re-entrenar por fold
- Evaluar y comparar el desempeño final frente al baseline.


### **8.3. Generación de espacio de búsqueda de HP**

#### **8.3.1. Explicación de `suggest_hparams_fine(trial)`**

A partir de los hiperparámetros obtenidos en el **coarse tuning**, el objetivo del *fine tuning* es **refinar localmente** el espacio de búsqueda, evitando explorar regímenes de entrenamiento ya descartados.  
En esta etapa **no se vuelven a tunear parámetros de arquitectura**, sino únicamente aquellos asociados a la dinámica de entrenamiento y regularización.

##### Rangos recomendados para *fine tuning*

Los rangos se definen **centrados en los valores óptimos del coarse**, con variaciones controladas:

###### 🔹 Learning rate (`lr ≈ 4.52e-4`)
- Rango:
  - `lr ∈ [lr₀ × 0.6 , lr₀ × 1.6]`
  - Aproximadamente: **[2.7e-4 , 7.2e-4]**
- Justificación:
  - Permite ajustar la velocidad de convergencia sin cambiar de régimen de optimización.

######  🔹 Weight decay (`wd ≈ 8.72e-6`)
- Rango:
  - `weight_decay ∈ [wd₀ × 0.5 , wd₀ × 2.0]`
  - Aproximadamente: **[4.4e-6 , 1.74e-5]**
- Justificación:
  - Mantiene la regularización en el mismo orden de magnitud.
  - Evita explorar valores demasiado grandes que ya corresponden a un tuning grueso.

######  🔹 Gradient clipping (`grad_clip ≈ 1.66`)
- Rango recomendado:
  - Absoluto: **[1.2 , 2.0]**
  - Alternativamente: `[gc₀ × 0.8 , gc₀ × 1.2] → [1.33 , 1.99]`
- Justificación:
  - Ajuste fino de estabilidad numérica sin introducir sesgos fuertes en el gradiente.

###### 🔹 Dropout (`dropout ≈ 0.143`)
- Rango:
  - `dropout ∈ [0.08 , 0.22]`
- Justificación:
  - Regularización moderada alrededor del valor óptimo encontrado.
  - Evita underfitting (dropout muy alto) o sobreajuste (dropout muy bajo).

###### 🔹 Head dropout (`head_dropout ≈ 0.116`)
- Rango:
  - `head_dropout ∈ [0.05 , 0.22]`
- Recomendación adicional:
  - Mantener coherencia: `head_dropout ≤ dropout + 0.05`
- Justificación:
  - Ajuste fino de la regularización específica del *regression head*.

######  Consideraciones finales
- En *fine tuning*:
  - **Arquitectura fija** (d_model, capas, heads, pooling).
  - **Espacios de búsqueda acotados**.
  - **Mayor número de épocas y paciencia** que en el coarse (por ejemplo, `max_epochs ≈ 30`, `patience ≈ 8`) para permitir convergencia fina.
- Este enfoque reduce ruido en Optuna y maximiza la probabilidad de mejoras marginales reales respecto al baseline.

#### **8.3.2. Código `suggest_hparams_fine(trial)`**

In [87]:
import optuna

# =========================================
# ESPACIO DE BÚSQUEDA (Optuna) — FINE TUNING
# Refina alrededor de coarse_best_params
# =========================================
def suggest_hparams_fine(trial):
    base = coarse_best_params          # dict plano (ya cargado)
    hp = dict(base)                    # copia

    # -------------------------
    # 1) Entrenamiento (rangos estrechos alrededor del coarse)
    # -------------------------

    #'lr': 4.52e-4
    #lr ∈ [lr0 * 0.6, lr0 * 1.6] → aprox [2.7e-4, 7.2e-4]
    lr0 = float(base["lr"])
    hp["lr"] = trial.suggest_float("lr", lr0 * 0.6, lr0 * 1.6, log=True)

    #'weight_decay': 8.72e-6
    #wd ∈ [wd0 * 0.5, wd0 * 2.0] → aprox [4.4e-6, 1.74e-5]
    wd0 = float(base["weight_decay"])
    hp["weight_decay"] = trial.suggest_float("weight_decay", wd0 * 0.5, wd0 * 2.0, log=True)

    #'grad_clip': 1.66
    #gc ∈ [1.2, 2.0] (rango absoluto) o [gc0*0.8, gc0*1.2] → [1.33, 1.99]
    gc0 = float(base["grad_clip"])
    hp["grad_clip"] = trial.suggest_float("grad_clip", gc0 * 0.8, gc0 * 1.2)

    # -------------------------
    # 2) Regularización (ajuste fino)
    # -------------------------

    #'dropout': 0.143
    #dropout ∈ [0.08, 0.22] (acotado y razonable)
    d0 = float(base["dropout"])
    hp["dropout"] = trial.suggest_float("dropout", max(0.0, d0 - 0.06), min(0.30, d0 + 0.06))

    #'head_dropout': 0.116
    #head_dropout ∈ [0.05, 0.22]       # (y opcionalmente forzar head_dropout <= dropout + 0.05 para coherencia)
    hd0 = float(base["head_dropout"])
    hp["head_dropout"] = trial.suggest_float("head_dropout", max(0.0, hd0 - 0.06), min(0.30, hd0 + 0.06))

    # -------------------------
    # 3) Arquitectura y pooling (FIJOS = coarse)
    # -------------------------
    hp["d_model"]    = int(base["d_model"])
    hp["n_layers"]   = int(base["n_layers"])
    hp["n_heads"]    = int(base["n_heads"])
    hp["ff_mult"]    = int(base["ff_mult"])
    hp["activation"] = base["activation"]
    hp["pooling"]    = base["pooling"]

    # dim_feedforward (si tu builder lo usa)
    hp["dim_feedforward"] = hp["d_model"] * hp["ff_mult"]

    # Compatibilidad MultiHeadAttention
    if hp["d_model"] % hp["n_heads"] != 0:
        raise optuna.TrialPruned()

    # -------------------------
    # 4) Defaults (si no los estás seteando fuera)
    # -------------------------
    hp.setdefault("batch_size", 256)
    hp.setdefault("use_amp", True)
    hp.setdefault("max_epochs", 30)
    hp.setdefault("patience", 8)

    return hp


#### **8.4.Builder del modelo – Coarse Tuning (`build_transformer_model_coarse`)**

#### **8.4.1. Conceptos**

Esta función construye el modelo **end-to-end** (Encoder + Pooling + Regression Head) que se utiliza durante la etapa de **coarse tuning con Optuna**.

Su objetivo es garantizar que **todos los trials** se entrenen bajo una arquitectura consistente, variando únicamente los hiperparámetros definidos en el espacio de búsqueda.

1. **Construye el Transformer Encoder (`TimeSeriesEncoder`)**  
   Utiliza los hiperparámetros propuestos por Optuna (`d_model`, `n_heads`, `n_layers`, `dropout`, `activation`, etc.) para definir la capacidad y profundidad del modelo.

2. **Aplica pooling temporal (`TemporalPooling`)**  
   Resume la secuencia temporal completa en un vector fijo usando el método seleccionado (`mean` o `last`).

3. **Define el Head de regresión (`RegressionHead`)**  
   Transforma la representación final en una predicción escalar, usando la **misma función de activación que el encoder**, decisión intencional para reducir la complejidad del espacio de búsqueda durante el coarse tuning.

4. **Ensambla el modelo completo**  
   Devuelve tanto el modelo `nn.Sequential` como cada submódulo por separado, permitiendo reutilización en etapas de predicción o evaluación.



**Nota de diseño (importante)**

Durante el **coarse tuning**:
- Se utiliza **una única activación (`relu` o `gelu`)** para todo el modelo.
- Esto reduce el espacio de búsqueda y mejora la estabilidad del proceso de optimización.
- La separación de activaciones entre encoder y head queda reservada para una etapa posterior de *fine tuning*.

####**8.4.2. Código `build_transformer_model_fine**

In [96]:
import torch.nn as nn

def build_transformer_model_fine(hp: dict, F: int, device: str):
    encoder = TimeSeriesEncoder(
        input_dim=F,
        d_model=hp["d_model"],
        nhead=hp["n_heads"],
        num_layers=hp["n_layers"],
        dim_feedforward=hp.get("dim_feedforward", hp["d_model"] * 2),
        dropout=hp["dropout"],
        activation=hp.get("activation", "gelu"),
    ).to(device)

    pool = TemporalPooling(hp.get("pooling", "mean")).to(device)

    head = RegressionHead(
        d_model=hp["d_model"],
        dropout=hp.get("head_dropout", hp["dropout"]),
        activation=hp.get("activation", "gelu"),
    ).to(device)

    model = nn.Sequential(encoder, pool, head).to(device)
    return model, encoder, head, pool

### **8.5.Runner de entrenamiento por folds (Coarse Tuning)**

#### 8.5.1. Concepto

Este runner entrena y evalúa el modelo sobre **todos los folds** utilizando un único conjunto de hiperparámetros (`hp`), y es invocado por Optuna desde la función `objective(trial)`.

Esta función ejecuta un ciclo completo de entrenamiento y evaluación **por fold** para un conjunto fijo de hiperparámetros (`hp`):

1. Carga los datos correspondientes a cada fold.
2. Reconstruye los `DataLoader` usando el batch size definido en `hp`.
3. Construye el modelo completo (encoder + pooling + head) a partir de los
   hiperparámetros propuestos por Optuna.
4. Entrena el modelo usando el trainer unificado (mismo loop que el baseline).
5. Genera predicciones para train, valid y test.
6. Calcula métricas por split y por fold.
7. Guarda checkpoints completos con:
   - pesos del modelo
   - métricas
   - scaler de `y`
   - hiperparámetros reales usados en el trial.

El resultado es un conjunto de métricas **comparables entre folds**, que luego
Optuna utiliza para evaluar la calidad del trial.

#### 8.5.2. Código

In [89]:
# ----------------------------------------------------------------------------------
# NOTA DE DISEÑO
# ----------------------------------------------------------------------------------
# Este runner NO implementa lógica de Optuna.
# Su única responsabilidad es entrenar y evaluar todos los folds para un conjunto
# fijo de hiperparámetros (hp).
#
# Optuna interactúa únicamente a través de la función objective(trial), que:
#   - llama a este runner
#   - agrega la lógica de agregación de métricas entre folds
#   - reporta el valor final del trial a Optuna
#
# Esto mantiene una separación clara entre:
#   - entrenamiento / evaluación (este runner)
#   - optimización de hiperparámetros (Optuna)
# ----------------------------------------------------------------------------------

import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

# Tamaño de ventana y cantidad de features
T = windows_size
F = len(features_90)

# -----------------------------------------
# Runner: entrena TODOS los folds con un hp
# -----------------------------------------
def run_folds_with_hp(hp, save_ckpt: bool = True, skip_if_exists: bool = False):
    """
    Entrena y evalúa por fold usando los hiperparámetros hp.
    Devuelve métricas por fold (train/valid/test).
    """

    models      = {}
    scalers_y   = {}
    ytr_pred_d  = {}
    yva_pred_d  = {}
    yte_pred_d  = {}

    metrics_by_fold = {}

    for fold in k_folds:
        model_key = f"fine_fold_{fold}"

        # (opcional) saltar si ya existe en tu DF
        if skip_if_exists and ("fine_folds_metrics" in globals()
            and fine_folds_metrics is not None
            and model_key in fine_folds_metrics.index):
            print(f"Omitimos: {model_key} ya existe en fine_folds_metrics")
            continue

        print(f"\n=== Entrenando (HP) {model_key} ===")

        # -------------------------
        # 1) Datos del fold
        # -------------------------
        Xtr_3d = Xtr[fold]; Xva_3d = Xva[fold]; Xte_3d = Xte[fold]
        ytr = y_train_sc[fold]; yva = y_valid_sc[fold]; yte = y_test_sc[fold]

        # Flatten 3D -> 2D para tu make_loaders
        Xtr_flat = Xtr_3d.reshape(Xtr_3d.shape[0], -1)
        Xva_flat = Xva_3d.reshape(Xva_3d.shape[0], -1)
        Xte_flat = Xte_3d.reshape(Xte_3d.shape[0], -1)

        # -------------------------
        # 2) Scaler y del fold
        # -------------------------
        scaler_y_fold = get_y_scaler(ytr)

        # -------------------------
        # 3) DataLoaders (bs tunable)
        # -------------------------
        dl_tr_fold, dl_va_fold = make_loaders(
            Xtr_flat, ytr,
            Xva_flat, yva,
            T=T, F=F,
            y_scaler=scaler_y_fold,
            bs=hp["batch_size"],
        )

        # -------------------------
        # 4) Construir modelo desde hp (arquitectura + pooling)
        # -------------------------
        model_fold, encoder_fold, head_fold, pool = build_transformer_model_fine(
            hp=hp, F=F, device=device
        )

        # -------------------------
        # 5) Entrenar con hp (no base_hparams)
        # -------------------------
        model_fold, best_val = train_model(
            model_fold, dl_tr_fold, dl_va_fold,
            device=device,
            lr=hp["lr"],
            weight_decay=hp["weight_decay"],
            max_epochs=hp["max_epochs"],
            patience=hp["patience"],
            grad_clip=hp["grad_clip"],
            use_amp=hp["use_amp"],
        )

        # -------------------------
        # 6) Predicciones train/valid/test
        # -------------------------
        ytr_pred = predict_set(encoder_fold, pool, head_fold, Xtr_flat, T, F, device,
                               batch_size=4096, y_scaler=scaler_y_fold)

        yva_pred = predict_set(encoder_fold, pool, head_fold, Xva_flat, T, F, device,
                               batch_size=4096, y_scaler=scaler_y_fold)

        yte_pred = predict_set(encoder_fold, pool, head_fold, Xte_flat, T, F, device,
                               batch_size=4096, y_scaler=scaler_y_fold)

        # -------------------------
        # 7) Métricas
        # -------------------------
        metrics_tr = evaluate_model(None, None, y_true=ytr, y_pred=ytr_pred)
        metrics_va = evaluate_model(None, None, y_true=yva, y_pred=yva_pred)
        metrics_te = evaluate_model(None, None, y_true=yte, y_pred=yte_pred)

        metrics_by_fold[fold] = {
            "train": metrics_tr,
            "valid": metrics_va,
            "test":  metrics_te,
        }

        # (opcional) guardar en DF global
        if "fine_folds_metrics" in globals() and fine_folds_metrics is not None:
            for split, m in [("train", metrics_tr), ("valid", metrics_va), ("test", metrics_te)]:
                for k, v in m.items():
                    fine_folds_metrics.loc[model_key, f"{split}_{k}"] = v

        # -------------------------
        # 8) Guardar en RAM
        # -------------------------
        models[fold]     = model_fold
        scalers_y[fold]  = scaler_y_fold
        ytr_pred_d[fold] = ytr_pred
        yva_pred_d[fold] = yva_pred
        yte_pred_d[fold] = yte_pred

        # -------------------------
        # 9) Guardar checkpoint (con hp reales)
        # -------------------------
        if save_ckpt:
            ruta_ckpt = path_models_fine[fold]["model_path"]

            checkpoint = {
                "model_state":   model_fold.state_dict(),
                "encoder_state": encoder_fold.state_dict(),
                "head_state":    head_fold.state_dict(),
                "scaler_y":      scaler_y_fold,
                "metrics_train": metrics_tr,
                "metrics_valid": metrics_va,
                "metrics_test":  metrics_te,
                "hparams": {
                    "T": T,
                    "F": F,
                    "lr": hp["lr"],
                    "weight_decay": hp["weight_decay"],
                    "max_epochs": hp["max_epochs"],
                    "patience": hp["patience"],
                    "grad_clip": hp["grad_clip"],
                    "use_amp": hp["use_amp"],
                    "pooling": hp.get("pooling", "mean"),
                    "batch_size": hp["batch_size"],
                    "d_model": hp["d_model"],
                    "n_heads": hp["n_heads"],
                    "n_layers": hp["n_layers"],
                    "dim_feedforward": hp.get("dim_feedforward"),
                    "dropout": hp["dropout"],
                    "activation": hp.get("activation", "gelu"),
                    "head_dropout": hp.get("head_dropout", hp["dropout"]),
                },
            }

            torch.save(checkpoint, ruta_ckpt)
            print(f"✔ Checkpoint guardado fold {fold}: {ruta_ckpt}")

        # limpieza ligera
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return metrics_by_fold, models, scalers_y, ytr_pred_d, yva_pred_d, yte_pred_d


### **8.6. Rutas para guardar modelos coarse_tuning por fold**

In [90]:
import os

# --- Función unificada ---
def ruta_modelo_fold(fold: int, subcarpeta: str) -> dict:
    """
    Crea la carpeta destino y devuelve la ruta completa
    para almacenar el modelo correspondiente al fold.
    """
    base = f"{drive_path}/5_transformer_model/{subcarpeta}"
    os.makedirs(base, exist_ok=True)

    model_path = os.path.join(base, f"fine_fold_{fold}.pt")
    return {"model_path": model_path}

In [91]:
# --- Generar diccionario de rutas ---
subcarpeta = "5_5_fine_tuning"
path_models_fine = {}

for k in k_folds:
    path_models_fine[k] = ruta_modelo_fold(k, subcarpeta)

# --- Resultado final ---
path_models_fine

{1: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_1.pt'},
 2: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_2.pt'},
 3: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_3.pt'},
 4: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_4.pt'},
 5: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_5.pt'}}

## **9. Función objetivo para Fine Hyperparameter Tuning**

La siguiente función define la función objetivo utilizada por Optuna para realizar el tuneo grueso (coarse tuning) de hiperparámetros del modelo Transformer. Su propósito es explorar un espacio amplio de hiperparámetros de forma estable y reproducible, manteniendo una evaluación estrictamente comparable con el modelo baseline.

### **9.1. Descripción general**

La función recibe un objeto trial de Optuna, donde para cada trial:

- Se muestrean hiperparámetros mediante suggest_hparams.
- Se entrena el modelo en todos los folds definidos.
- Se evalúa el desempeño usando RMSE en escala original (raw) sobre el conjunto de validación.

El score final del trial es el promedio del RMSE de validación entre folds.

####**Detalles clave del diseño del objective_coarse**

##### 1. Muestreo de hiperparámetros

En cada *trial*, Optuna explora un conjunto amplio de hiperparámetros que abarcan tanto la **arquitectura del modelo** como la **dinámica de entrenamiento**:

- **Arquitectura del Transformer**
  - `d_model`
  - `n_layers`
  - `n_heads`
  - `pooling`

- **Regularización**
  - `dropout`
  - `weight_decay`

- **Optimización**
  - `lr`
  - `grad_clip`

- **Parámetros de entrenamiento**
  - `batch_size`
  - `max_epochs`
  - `patience`

Este enfoque permite identificar regiones prometedoras del espacio de búsqueda sin restringir prematuramente la complejidad del modelo.

---

##### 2. Entrenamiento por *K-Folds*

Para cada *fold* del esquema de validación cruzada:

- Se reconstruyen las ventanas de entrada a partir de los datos aplanados.
- Se ajusta un `StandardScaler` **exclusivamente con `y_train` del fold**, evitando cualquier tipo de *data leakage*.
- El `DataLoader` se configura con `num_workers = 0`, con el objetivo de:
  - Reducir ruido estocástico entre *trials*.
  - Mejorar la estabilidad del proceso de tuning.
  - Garantizar reproducibilidad de los resultados.

---

##### 3. Entrenamiento del modelo

- El entrenamiento se realiza mediante la función `train_model`.
- Internamente se aplica:
  - *Early stopping* basado en validación.
  - *Pruning por época* a través de Optuna.
- El valor `best_val_mse` devuelto corresponde a la pérdida en **escala normalizada (z-score)**, pero **no se utiliza como métrica final del trial**.

---

##### 4. Evaluación estrictamente comparable con el baseline

Para asegurar una comparación justa y directa con el modelo baseline:

- Las predicciones de validación se generan con `predict_set`, aplicando `inverse_transform` para volver a escala original.
- Las métricas se calculan sobre valores **raw** utilizando `evaluate_model`.
- El score del fold se define como:

> **RMSE en escala original (raw) sobre validación**

De este modo, Optuna optimiza exactamente la misma métrica utilizada en los experimentos base.

---

##### 5. Agregación y score final del trial

- El score final del trial es el **promedio del RMSE(raw)** obtenido en todos los folds.
- Este valor:
  - Se devuelve a Optuna como objetivo de optimización.
  - Se almacena como `user_attr` (`mean_valid_rmse_raw`) para trazabilidad y análisis posterior del estudio.

---

##### **Resultado esperado**

Este diseño permite:

- Identificar hiperparámetros robustos en una fase de exploración gruesa.
- Evitar sesgos introducidos por métricas en escala normalizada o paralelismo en los loaders.
- Reutilizar los mejores hiperparámetros en un **reentrenamiento final directamente comparable con el baseline**.



###**9.2. Código `objective_fine(trial)`**

In [92]:
device = "cuda" if torch.cuda.is_available() else "cpu"

import numpy as np
import gc
import torch
import optuna

def objective_fine(trial):
    if trial.number == 0:
        print("▶ Device usado:", device)

    print(f"\n==============================")
    print(f"▶ Trial {trial.number} | START (FINE)")
    print(f"==============================")

    # --------------------------------------------------
    # 1) Sampleo de hiperparámetros
    # --------------------------------------------------
    hp = suggest_hparams_fine(trial)

    print(
        f"▶ Hiperparámetros Trial {trial.number} | "
        f"bs={hp['batch_size']} | "
        f"max_epochs={hp['max_epochs']} | "
        f"patience={hp['patience']} | "
        f"lr={hp['lr']:.2e} | "
        f"d_model={hp['d_model']} | "
        f"n_layers={hp['n_layers']} | "
        f"n_heads={hp['n_heads']} | "
        f"pool={hp['pooling']}"
    )

    valid_rmses = []

    # --------------------------------------------------
    # 2) Loop por folds
    # --------------------------------------------------
    for fold in k_folds:
        print(f"\n  → Trial {trial.number} | Fold {fold} | training...")

        # -------------------------
        # Datos del fold
        # -------------------------
        Xtr_3d = Xtr[fold]
        Xva_3d = Xva[fold]
        ytr = y_train_sc[fold]
        yva = y_valid_sc[fold]

        Xtr_flat = Xtr_3d.reshape(Xtr_3d.shape[0], -1)
        Xva_flat = Xva_3d.reshape(Xva_3d.shape[0], -1)

        # -------------------------
        # Scaler del target
        # -------------------------
        scaler_y_fold = get_y_scaler(ytr)

        # -------------------------
        # DataLoaders
        # -------------------------
        dl_tr_fold, dl_va_fold = make_loaders(
            Xtr_flat, ytr,
            Xva_flat, yva,
            T=T, F=F,
            y_scaler=scaler_y_fold,
            bs=hp["batch_size"],
            num_workers=0,
            pin_memory=True,
            persistent_workers=False,
        )

        # -------------------------
        # Modelo (builder fine)
        # -------------------------
        model_fold, encoder_fold, head_fold, pool = build_transformer_model_fine(
            hp=hp, F=F, device=device
        )

        # -------------------------
        # Entrenamiento (pruning por época dentro de train_model)
        # Nota: el best_val devuelto corresponde al MSE en z-score,
        # pero NO lo usamos como score final porque queremos ser 100%
        # comparables con el baseline (evaluate_model sobre y raw).
        # -------------------------
        model_fold, best_val_mse = train_model(
            model_fold, dl_tr_fold, dl_va_fold,
            device=device,
            lr=hp["lr"],
            weight_decay=hp["weight_decay"],
            max_epochs=hp["max_epochs"],
            patience=hp["patience"],
            grad_clip=hp["grad_clip"],
            use_amp=hp["use_amp"],
            trial=trial,
            prune_after_epoch = 7,
        )

        # -------------------------
        # Score del fold (idéntico al baseline):
        #  - predict_set devuelve y_pred en RAW (inverse_transform)
        #  - evaluate_model compara RAW vs RAW
        # -------------------------
        yva_pred = predict_set(
            encoder_fold, pool, head_fold,
            Xva_flat, T, F,
            device,
            batch_size=4096,
            y_scaler=scaler_y_fold
        )

        rmse_raw = float(evaluate_model(None, None, y_true=yva, y_pred=yva_pred)["RMSE"])
        valid_rmses.append(rmse_raw)

        mean_rmse_so_far = float(np.mean(valid_rmses))
        print(
            f"  ✔ Fold {fold} | "
            f"valid_RMSE(raw)={rmse_raw:.6f} | "
            f"mean_RMSE(raw)={mean_rmse_so_far:.6f}"
        )


        # -------------------------
        # Pruning adicional por fold
        # TAL VEZ LO USEMOS EN EL TUNING FINO
        # -------------------------
        #trial.report(mean_rmse_so_far, step=100 + fold)
        #if trial.should_prune():
        #    raise optuna.TrialPruned()

        # -------------------------
        # Limpieza de memoria
        # -------------------------
        del model_fold, encoder_fold, head_fold, pool
        del dl_tr_fold, dl_va_fold
        del Xtr_flat, Xva_flat, Xtr_3d, Xva_3d
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------
    # 3) Score final del trial
    # --------------------------------------------------
    score = float(np.mean(valid_rmses))
    trial.set_user_attr("mean_valid_rmse_raw", score)

    print(f"\n■ Trial {trial.number} | SCORE (mean_valid_RMSE(raw)) = {score:.6f}")
    return score

###**9.3. Ejecución `objective_fine(trial)`**

In [93]:
import os, json, joblib
import pandas as pd

# ---------------------------------------------
# 0) Chequeo: artefactos de Optuna (solo estudio)
# ---------------------------------------------
study_exists = (
    os.path.exists(paths_study["study_pkl"]) and
    os.path.exists(paths_study["best_json"]) and
    os.path.exists(paths_study["trials_csv"])
)

if study_exists:
    print("✔ Fine tuning ya existe. Cargando artefactos de Optuna...")

    study = joblib.load(paths_study["study_pkl"])

    with open(paths_study["best_json"], "r", encoding="utf-8") as f:
        best_payload = json.load(f)

    trials_df = pd.read_csv(paths_study["trials_csv"])

    # best_payload puede ser:
    #  (A) dict simple de best_params (mi versión anterior)
    #  (B) payload completo con best_trial_number/best_value/best_params (tu versión)
    if isinstance(best_payload, dict) and "best_params" in best_payload:
        print("Best trial:", best_payload.get("best_trial_number", study.best_trial.number))
        print("Best value:", best_payload.get("best_value", study.best_value))
        best_params = best_payload["best_params"]
    else:
        print("Best trial:", getattr(study.best_trial, "number", None))
        print("Best value:", study.best_value)
        best_params = best_payload  # dict simple

    print("Best params:")
    for k, v in best_params.items():
        print(f"  - {k}: {v}")

else:
    print("Fine tuning NO encontrado. Se ejecutará Optuna.")

    import optuna
    optuna.logging.set_verbosity(optuna.logging.INFO)

    sampler = optuna.samplers.TPESampler(seed=42)
    #pruner  = optuna.pruners.SuccessiveHalvingPruner()
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=7)      # menos agresivo

    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    study.optimize(objective_fine, n_trials=30, gc_after_trial=True)

    print("\n✔ Optuna finalizado.")
    print("Best value:", study.best_value)
    print("Best params:", study.best_params)

    # ---------------------------------------------
    # Guardar artefactos de Optuna (estudio)
    # ---------------------------------------------
    joblib.dump(study, paths_study["study_pkl"])

    # Guardado robusto: payload completo (recomendado)
    payload = {
        "best_trial_number": int(study.best_trial.number),
        "best_value": float(study.best_value),
        "best_params": study.best_params,
    }
    with open(paths_study["best_json"], "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    trials_df = study.trials_dataframe()
    trials_df.to_csv(paths_study["trials_csv"], index=False)

    print("\n✔ Artefactos Optuna guardados:")
    print(" -", paths_study["study_pkl"])
    print(" -", paths_study["best_json"])
    print(" -", paths_study["trials_csv"])

[I 2025-12-20 22:31:46,898] A new study created in memory with name: no-name-00c2a7a4-caa6-41fa-b81a-4263b618b4b8


Fine tuning NO encontrado. Se ejecutará Optuna.
▶ Device usado: cuda

▶ Trial 0 | START (FINE)
▶ Hiperparámetros Trial 0 | bs=256 | max_epochs=30 | patience=8 | lr=3.92e-04 | d_model=64 | n_layers=4 | n_heads=8 | pool=last

  → Trial 0 | Fold 1 | training...
  ✔ Fold 1 | valid_RMSE(raw)=0.004100 | mean_RMSE(raw)=0.004100

  → Trial 0 | Fold 2 | training...
  ✔ Fold 2 | valid_RMSE(raw)=0.002647 | mean_RMSE(raw)=0.003373

  → Trial 0 | Fold 3 | training...
  ✔ Fold 3 | valid_RMSE(raw)=0.001987 | mean_RMSE(raw)=0.002911

  → Trial 0 | Fold 4 | training...
  ✔ Fold 4 | valid_RMSE(raw)=0.002056 | mean_RMSE(raw)=0.002697

  → Trial 0 | Fold 5 | training...
  ✔ Fold 5 | valid_RMSE(raw)=0.001860 | mean_RMSE(raw)=0.002530


[I 2025-12-20 22:43:55,606] Trial 0 finished with value: 0.0025299711197509004 and parameters: {'lr': 0.00039198382720712955, 'weight_decay': 1.6297089596524996e-05, 'grad_clip': 1.8122580949857792, 'dropout': 0.15518674011531647, 'head_dropout': 0.07424068648367216}. Best is trial 0 with value: 0.0025299711197509004.



■ Trial 0 | SCORE (mean_valid_RMSE(raw)) = 0.002530

▶ Trial 1 | START (FINE)
▶ Hiperparámetros Trial 1 | bs=256 | max_epochs=30 | patience=8 | lr=3.16e-04 | d_model=64 | n_layers=4 | n_heads=8 | pool=last

  → Trial 1 | Fold 1 | training...
  ✔ Fold 1 | valid_RMSE(raw)=0.004018 | mean_RMSE(raw)=0.004018

  → Trial 1 | Fold 2 | training...
  ✔ Fold 2 | valid_RMSE(raw)=0.002690 | mean_RMSE(raw)=0.003354

  → Trial 1 | Fold 3 | training...
  ✔ Fold 3 | valid_RMSE(raw)=0.001982 | mean_RMSE(raw)=0.002897

  → Trial 1 | Fold 4 | training...
  ✔ Fold 4 | valid_RMSE(raw)=0.002049 | mean_RMSE(raw)=0.002685

  → Trial 1 | Fold 5 | training...
  ✔ Fold 5 | valid_RMSE(raw)=0.001915 | mean_RMSE(raw)=0.002531


[I 2025-12-20 22:56:43,996] Trial 1 finished with value: 0.0025307785399514447 and parameters: {'lr': 0.0003163548939992766, 'weight_decay': 4.72816719173229e-06, 'grad_clip': 1.9012673645944478, 'dropout': 0.15548152342085714, 'head_dropout': 0.14048715896610525}. Best is trial 0 with value: 0.0025299711197509004.


...


▶ Trial 28 | START (FINE)
▶ Hiperparámetros Trial 28 | bs=256 | max_epochs=30 | patience=8 | lr=2.95e-04 | d_model=64 | n_layers=4 | n_heads=8 | pool=last

  → Trial 28 | Fold 1 | training...
  ✔ Fold 1 | valid_RMSE(raw)=0.003846 | mean_RMSE(raw)=0.003846

  → Trial 28 | Fold 2 | training...
  ✔ Fold 2 | valid_RMSE(raw)=0.002640 | mean_RMSE(raw)=0.003243

  → Trial 28 | Fold 3 | training...
  ✔ Fold 3 | valid_RMSE(raw)=0.001964 | mean_RMSE(raw)=0.002817

  → Trial 28 | Fold 4 | training...
  ✔ Fold 4 | valid_RMSE(raw)=0.002025 | mean_RMSE(raw)=0.002619

  → Trial 28 | Fold 5 | training...
  ✔ Fold 5 | valid_RMSE(raw)=0.001966 | mean_RMSE(raw)=0.002488


[I 2025-12-21 00:40:59,842] Trial 28 finished with value: 0.002488186357624136 and parameters: {'lr': 0.00029518702676514005, 'weight_decay': 4.485625478973666e-06, 'grad_clip': 1.834957917755345, 'dropout': 0.11773144845235464, 'head_dropout': 0.0906228567093325}. Best is trial 26 with value: 0.002487750845742693.



■ Trial 28 | SCORE (mean_valid_RMSE(raw)) = 0.002488

▶ Trial 29 | START (FINE)
▶ Hiperparámetros Trial 29 | bs=256 | max_epochs=30 | patience=8 | lr=3.40e-04 | d_model=64 | n_layers=4 | n_heads=8 | pool=last

  → Trial 29 | Fold 1 | training...


[I 2025-12-21 00:42:02,643] Trial 29 pruned. 



✔ Optuna finalizado.
Best value: 0.002487750845742693
Best params: {'lr': 0.0003302857650370811, 'weight_decay': 6.372757695656055e-06, 'grad_clip': 1.7205976274277908, 'dropout': 0.11722521501791065, 'head_dropout': 0.06912576493333449}

✔ Artefactos Optuna guardados:
 - /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/optuna_study/fine_study.pkl
 - /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/optuna_study/fine_best_params.json
 - /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/optuna_study/fine_trials.csv


###**9.4. Resultados de `objective_fine(trial)`**

**Mejor trial (Fine):**
- **Best value (mean valid RMSE raw):** `0.002487750845742693`

**Mejores hiperparámetros (Fine):**
- `lr`: `0.0003302857650370811`
- `weight_decay`: `6.372757695656055e-06`
- `grad_clip`: `1.7205976274277908`
- `dropout`: `0.11722521501791065`
- `head_dropout`: `0.06912576493333449`

### Comentarios rápidos
- El **fine tuning** logró una **mejora marginal** respecto al coarse (consistente con lo esperable cuando el espacio de búsqueda ya está acotado).
- Se observa una **reducción de regularización**:
  - `dropout` y `head_dropout` bajaron (modelo ligeramente menos regularizado).
- El `lr` quedó en un rango moderado (más bajo que el coarse), típico de ajuste fino.
- `weight_decay` se mantiene muy bajo (regularización L2 suave).
- `grad_clip` quedó cercano al coarse, lo que sugiere que el control de gradiente sigue siendo necesario y estable.

## **10. Re-entrenamiento del modelo baseline con `fine_best_params`**

### **10.1 Selección de los mejores hiperparámetros (Fine Tuning)**

En este punto se recuperan los hiperparámetros óptimos obtenidos a partir del estudio de Optuna durante la etapa de *coarse tuning*. Estos valores corresponden al *trial* con menor error promedio de validación (RMSE en escala *raw*), calculado de forma consistente con el modelo baseline.

A partir de aquí, estos hiperparámetros se consideran **fijos** y se utilizarán para el re-entrenamiento completo del modelo baseline por *fold*, permitiendo una comparación directa y justa frente al baseline original.

In [94]:
# Recuperar los mejores hiperparámetros del coarse tuning
fine_best_params = study.best_params
fine_best_params

{'lr': 0.0003302857650370811,
 'weight_decay': 6.372757695656055e-06,
 'grad_clip': 1.7205976274277908,
 'dropout': 0.11722521501791065,
 'head_dropout': 0.06912576493333449}

Combinación de hiperparametros

In [101]:
fine_best_params = {
  'lr': fine_best_params['lr'],
 'weight_decay': fine_best_params['weight_decay'],
 'grad_clip':  fine_best_params[ 'grad_clip'],
 'dropout': fine_best_params[ 'dropout'],
 'head_dropout': fine_best_params['head_dropout'],
 'd_model': coarse_best_params['d_model'],
 'n_layers': coarse_best_params['n_layers'],
 'n_heads': coarse_best_params['n_heads'],
 'ff_mult': coarse_best_params['ff_mult'],
 'activation': coarse_best_params[ 'activation'],
 'pooling': coarse_best_params['pooling'],
 'max_epochs': baseline_params['max_epochs'],
 'patience': baseline_params['patience'],
}


In [102]:
fine_best_params

{'lr': 0.0003302857650370811,
 'weight_decay': 6.372757695656055e-06,
 'grad_clip': 1.7205976274277908,
 'dropout': 0.11722521501791065,
 'head_dropout': 0.06912576493333449,
 'd_model': 64,
 'n_layers': 4,
 'n_heads': 8,
 'ff_mult': 4,
 'activation': 'relu',
 'pooling': 'last',
 'max_epochs': 50,
 'patience': 8}

### **10.2 Inicialización de los modelos FINE por fold**


Con los hiperparámetros óptimos ya definidos, se procede a instanciar la arquitectura del modelo para cada *fold* de la validación cruzada.  
En esta etapa **no se entrena el modelo**, sino que únicamente se construyen las instancias del Transformer (encoder + pooling + head) que luego serán entrenadas de forma independiente.

Crear un modelo por *fold* garantiza:
- Independencia total entre entrenamientos,
- Ausencia de fuga de información entre particiones,
- Reproducibilidad del proceso experimental.

In [103]:
# Diccionarios para almacenar los modelos y submódulos por fold
fine_model    = {}
fine_encoders = {}
fine_heads    = {}
fine_pool     = {}

# Inicialización de la arquitectura por fold
for k in k_folds:
    fine_model[k], fine_encoders[k], fine_heads[k], fine_pool[k] = (
        build_transformer_model_fine(
            hp=fine_best_params,
            F=F,
            device=device
        )
    )
    print(f"Modelo FINE creado para fold {k}")

Modelo FINE creado para fold 1
Modelo FINE creado para fold 2
Modelo FINE creado para fold 3
Modelo FINE creado para fold 4
Modelo FINE creado para fold 5


### **10.3 Re-entrenamiento FINE por fold (comparación directa con el baseline)**

En esta sección se re-entrena el modelo utilizando los **mejores hiperparámetros del finee tuning** (`finee_best_params`), pero manteniendo **idéntico el pipeline de entrenamiento y evaluación** respecto al baseline para asegurar comparabilidad.

#### **10.3.1. Descripción funcional**


El flujo por cada *fold* es:

1. **Carga de datos del fold** (`Xtr/Xva/Xte` y `ytr/yva/yte`) y re-formateo `3D → 2D` para compatibilidad con `make_loaders`.
2. **Ajuste del escalador del target** (`scaler_y_fold`) usando únicamente `y_train` del fold.
3. **Creación de DataLoaders** con configuración equivalente al baseline:
   - `batch_train = 256`
   - `num_workers = 8`, `prefetch_factor = 4`, `pin_memory = True`
4. **Selección del modelo del fold** ya instanciado con la arquitectura óptima (encoder + pooling + head).
5. **Entrenamiento** con `train_model` usando:
   - LR / weight_decay / grad_clip desde `coarse_best_params`
   - `max_epochs` y `patience` iguales al baseline (para mantener criterio de early stopping comparable)
6. **Predicción en train/valid/test** con `predict_set`, devolviendo `y_pred` en escala *raw* (inverse transform).
7. **Cálculo de métricas** mediante `evaluate_model` comparando `y_true(raw)` vs `y_pred(raw)`.
8. **Persistencia de resultados**:
   - métricas en `coarse_folds_metrics`
   - predicciones y escaladores en diccionarios en RAM
   - checkpoint por fold en disco (`model_state`, submódulos, `scaler_y_fold`, métricas e hiperparámetros)

Finalmente, se libera memoria explícitamente (RAM/GPU) para evitar acumulación entre folds.

#### **10.3.2. Código**

In [105]:
device = "cuda" if torch.cuda.is_available() else "cpu"

models    = {}
scalers_y  = {}
ytr_pred_d = {}
yva_pred_d = {}
yte_pred_d = {}

  # Tamaño de ventana y cantidad de features
T = windows_size # longitud de la ventana temporal
F = len(features_90) # número de features por paso
batch_train = 256 # batch size para entrenamiento
batch_pred = 8192

# Pooling compartido (sin parámetros entrenables)
#pool = TemporalPooling(coarse_best_params['pooling']).to(device)

# ------------------------------------------------------------
# Control global: entrenar o no según metrics_k_folds
# ------------------------------------------------------------
if flag_fine_folds_metrics:
      print("flag_fine_folds_metrics=True → se omite el entrenamiento de todos los folds.")
else:
      print("flag_fine_folds_metrics=False → se inicia entrenamiento por folds.")

      for fold in k_folds:
          #Definir una clave de modelo por fold (para registro de métricas)
          model_key = f"fine_fold_{fold}"

          # Si ya existe en la tabla de métricas, omitimos SOLO ese fold
          if ("fine_folds_metrics" in globals()
              and fine_folds_metrics is not None
              and model_key in fine_folds_metrics.index):
              print(f"Omitimos este entrenamiento: {model_key} ya existe en fine_folds_metrics")
              continue

          print(f"\n=== Entrenando modelo: {model_key} ===")

          # ------------------------------------------------------------
          # 2) Recuperar X e y del fold
          # ------------------------------------------------------------
          Xtr_3d = Xtr[fold]
          Xva_3d = Xva[fold]
          Xte_3d = Xte[fold]

          ytr = y_train_sc[fold]
          yva = y_valid_sc[fold]
          yte = y_test_sc[fold]

          # Aplanar 3D -> 2D
          Xtr_flat = Xtr_3d.reshape(Xtr_3d.shape[0], -1)
          Xva_flat = Xva_3d.reshape(Xva_3d.shape[0], -1)
          Xte_flat = Xte_3d.reshape(Xte_3d.shape[0], -1)

          # ------------------------------------------------------------
          # 3) Scaler de y (solo train)
          # ------------------------------------------------------------
          scaler_y_fold = get_y_scaler(ytr)

          # ------------------------------------------------------------
          # 4) DataLoaders
          # ------------------------------------------------------------
          dl_tr_fold, dl_va_fold = make_loaders(
              Xtr_flat, ytr,
              Xva_flat, yva,
              T=T, F=F,
              y_scaler=scaler_y_fold,
              bs=batch_train,
              num_workers=8,            #Idéntico a baseline
              prefetch_factor=4,        #Idéntico a baseline
              pin_memory=True,          #Idéntico a baseline
              persistent_workers=False  #Idéntico a baseline
          )

          # ------------------------------------------------------------
          # 5) Modelo del fold: Tomar encoder/pool/head del fold (coherentes entre sí)
          # ------------------------------------------------------------

          model_fold   = fine_model[fold].to(device)
          encoder_fold = fine_encoders[fold].to(device)
          head_fold    = fine_heads[fold].to(device)
          pool_fold    = fine_pool[fold].to(device)

          # ------------------------------------------------------------
          # 6) Entrenamiento
          # ------------------------------------------------------------
          model_fold, best_val = train_model(
              model_fold,
              dl_tr_fold,
              dl_va_fold,
              device=device,
              max_epochs=baseline_params['max_epochs'],
              patience=baseline_params['patience'],
              use_amp=True,
              lr=fine_best_params['lr'],
              weight_decay = fine_best_params['weight_decay'],
              grad_clip=fine_best_params['grad_clip'],
          )

          # ------------------------------------------------------------
          # 7) Predicciones
          # ------------------------------------------------------------
          ytr_pred = predict_set(
              encoder_fold, pool_fold, head_fold,
              Xtr_flat, T, F,
              device,
              batch_size=batch_pred,
              y_scaler=scaler_y_fold
          )

          yva_pred = predict_set(
              encoder_fold, pool_fold, head_fold,
              Xva_flat, T, F,
              device,
              batch_size=batch_pred,
              y_scaler=scaler_y_fold
          )

          yte_pred = predict_set(
              encoder_fold, pool_fold, head_fold,
              Xte_flat, T, F,
              device,
              batch_size=batch_pred,
              y_scaler=scaler_y_fold
          )

          # ------------------------------------------------------------
          # 8) Cálculo de métricas por fold (train / valid / test)
          #    Usamos evaluate_model, pasando y_pred explícitamente.
          # ------------------------------------------------------------
          metrics_tr = evaluate_model(None, None, y_true=ytr, y_pred=ytr_pred)
          metrics_va = evaluate_model(None, None, y_true=yva, y_pred=yva_pred)
          metrics_te = evaluate_model(None, None, y_true=yte, y_pred=yte_pred)

          # ------------------------------------------------------------
          # 9) Guardar métricas en el DataFrame global: baseline_folds_metrics
          # ------------------------------------------------------------
          if ("fine_folds_metrics" in globals()) and (fine_folds_metrics is not None):

              # (opcional pero recomendado) asegurar que exista la fila
              if model_key not in fine_folds_metrics.index:
                  fine_folds_metrics.loc[model_key, :] = None

              for split, m in [("train", metrics_tr), ("valid", metrics_va), ("test", metrics_te)]:
                  for k, v in m.items():
                      fine_folds_metrics.loc[model_key, f"{split}_{k}"] = v

          # ------------------------------------------------------------
          # 10) Guardar en RAM
          # ------------------------------------------------------------
          models[fold]     = model_fold
          scalers_y[fold]  = scaler_y_fold
          ytr_pred_d[fold] = ytr_pred
          yva_pred_d[fold] = yva_pred
          yte_pred_d[fold] = yte_pred

          # ------------------------------------------------------------
          # 11) Checkpoint en disco
          # ------------------------------------------------------------
          ruta_ckpt = path_models_fine[fold]["model_path"]

          checkpoint = {
              "model_state":   model_fold.state_dict(),
              "encoder_state": encoder_fold.state_dict(),
              "head_state":    head_fold.state_dict(),
              "scaler_y":      scaler_y_fold,
              "metrics_train": metrics_tr,
              "metrics_valid": metrics_va,
              "metrics_test":  metrics_te,
              "hparams": {
                  "T": T,
                  "F": F,
                  "lr": fine_best_params['lr'],
                  "weight_decay": fine_best_params['weight_decay'],
                  "max_epochs": baseline_params['max_epochs'],
                  "patience": baseline_params['patience'],
                  "grad_clip": fine_best_params['grad_clip'],
                  "use_amp": True,
                  "pooling": fine_best_params['pooling'],
                  "batch_size": batch_train,
              },
          }

          torch.save(checkpoint, ruta_ckpt)
          print(f"✔ Checkpoint guardado para fold {fold} en: {ruta_ckpt}")

          # ------------------------------------------------------------
          # 12) Limpieza explícita de memoria (GPU + RAM)
          # ------------------------------------------------------------
          import gc

          del dl_tr_fold, dl_va_fold
          del ytr_pred, yva_pred, yte_pred

          torch.cuda.empty_cache()
          gc.collect()

## 13minutos

flag_fine_folds_metrics=False → se inicia entrenamiento por folds.

=== Entrenando modelo: fine_fold_1 ===
✔ Checkpoint guardado para fold 1 en: /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_1.pt

=== Entrenando modelo: fine_fold_2 ===
✔ Checkpoint guardado para fold 2 en: /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_2.pt

=== Entrenando modelo: fine_fold_3 ===
✔ Checkpoint guardado para fold 3 en: /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_3.pt

=== Entrenando modelo: fine_fold_4 ===
✔ Checkpoint guardado para fold 4 en: /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_4.pt

=== Entrenando modelo: fine_fold_5 ===
✔ Checkpoint guardado para fold 5 en: /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/fine_fold_5.pt


## **11. Métricas de re-entrenamiento**

### **11.1. Gestión y limpieza de métricas COARSE por fold**

Antes de registrar las nuevas métricas del re-entrenamiento COARSE, se verifica si el DataFrame `coarse_folds_metrics` ya contiene resultados previos.

- Si `flag_coarse_folds_metrics = True`, se asume que las métricas ya existen y no se modifican.
- En caso contrario, se eliminan de forma segura las columnas base de métricas
  (`RMSE`, `MAE`, `R2`, `SMAPE`, `DirAcc`) **solo si están presentes**.

Este paso evita:
- duplicación de columnas,
- inconsistencias entre ejecuciones parciales,
- contaminación de resultados previos con métricas nuevas.

De esta manera, el DataFrame queda preparado para almacenar exclusivamente las métricas correspondientes al re-entrenamiento actual.


In [106]:
if flag_fine_folds_metrics:
    print("flag_fine_folds_metrics=True → ya existen métricas del entrenamiento.")
else:
    cols_base = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]

    # Eliminar solo las columnas que efectivamente existen
    cols_to_drop = [c for c in cols_base if c in fine_folds_metrics.columns]

    if cols_to_drop:
        fine_folds_metrics = fine_folds_metrics.drop(columns=cols_to_drop)

In [107]:
fine_folds_metrics

,train_RMSE,train_MAE,train_R2,train_SMAPE,train_DirAcc,valid_RMSE,valid_MAE,valid_R2,valid_SMAPE,valid_DirAcc,test_RMSE,test_MAE,test_R2,test_SMAPE,test_DirAcc
fine_fold_1,0.001810,0.001348,0.874424,75.601092,0.847722,0.003899,0.002727,0.561381,90.123125,0.807856,0.004715,0.002674,0.499991,113.471437,0.760520
fine_fold_2,0.002748,0.001948,0.725786,85.498130,0.812853,0.002582,0.001868,0.635192,85.033705,0.820628,0.004671,0.002537,0.509264,106.293365,0.793911
fine_fold_3,0.001742,0.001310,0.884249,71.927454,0.854936,0.001975,0.001466,0.677124,86.377377,0.819945,0.004526,0.002501,0.539245,99.285502,0.790572
fine_fold_4,0.002523,0.001795,0.739559,83.371419,0.825368,0.002142,0.001433,0.589607,92.841985,0.803920,0.004925,0.002632,0.454480,107.867149,0.797645
fine_fold_5,0.001720,0.001290,0.871136,74.039150,0.854532,0.001926,0.001415,0.677032,88.175465,0.812997,0.004576,0.002275,0.529180,89.753892,0.811360


Guardado de métricas

In [108]:
if flag_fine_folds_metrics == False:
  save_metrics(fine_folds_metrics, "5_5_fine_tuning","0_fine_folds_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

Métricas guardadas en /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/0_fine_folds_metrics.parquet


### **11.2. Cálculo de métricas ponderadas por conjunto**

Esta función calcula un **promedio ponderado de las métricas** considerando
simultáneamente los resultados de **train, valid y test** para cada *fold*.

La lógica es la siguiente:
1. Para cada métrica (`RMSE`, `MAE`, `R2`, `SMAPE`, `DirAcc`), se combinan los valores de `train`, `valid` y `test` usando pesos proporcionales al número de muestras (`w_train`, `w_valid`, `w_test`).
2. Se obtiene un valor ponderado por *fold*.
3. Finalmente, se promedian los valores ponderados entre todos los folds.

Este enfoque permite:
- reflejar el aporte relativo de cada conjunto según su tamaño,
- evitar que conjuntos pequeños distorsionen la métrica global,
- obtener una medida agregada más representativa del desempeño general del modelo.

In [109]:
def weighted_avg_metrics_from_df(df, w_train, w_valid, w_test):
    """
    Calcula el promedio ponderado de métricas a partir de un DataFrame
    con columnas del tipo train_RMSE, valid_RMSE, test_RMSE, etc.

    df : DataFrame con un fold por fila
    w_train, w_valid, w_test : pesos (cantidad de muestras por conjunto)
    """

    # Métricas base
    metricas = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]

    resultados = {}

    for m in metricas:
        col_train = f"train_{m}"
        col_valid = f"valid_{m}"
        col_test  = f"test_{m}"

        # Promedio ponderado por fold, luego promedio entre folds
        valores_fold = (
            df[col_train] * w_train +
            df[col_valid] * w_valid +
            df[col_test]  * w_test
        ) / (w_train + w_valid + w_test)

        # Promedio total final entre folds
        resultados[m] = valores_fold.mean()

    return resultados

En el siguiente bloque se construye el **dataset final de métricas COARSE**, a partir de las métricas detalladas de `train`, `valid` y `test` calculadas previamente.

La lógica es la siguiente:

  1. Se recuperan los pesos correspondientes (`w_train`, `w_valid`, `w_test`), proporcionales al número de muestras de cada conjunto.
  2. Se calcula el promedio ponderado de las métricas mediante      `weighted_avg_metrics_from_df`.
  3. Se almacenan los resultados finales en el DataFrame `coarse_metrics`, una fila por fold.

Este procedimiento permite resumir el desempeño del modelo en un **único conjunto de métricas comparables**, integrando de forma consistente la información de entrenamiento, validación y test.

In [110]:
if flag_fine_metrics:
    print("flag_fine_metrics=True → se omite el ponderado porque ya existe el dataset.")
else:
    print("flag_fine_metrics=False → se inicia el ponderado")
    for k in k_folds:
        w = pesos_folds[k]

        # promedio ponderado para ESTE fold (sale como dict)
        res = weighted_avg_metrics_from_df(
            fine_folds_metrics.loc[[f"fine_fold_{k}"]],
            w["w_train"],
            w["w_valid"],
            w["w_test"]
        )

        # índice correspondiente en baseline_metrics
        idx = f"fine_fold_{k}"

        # escribir directamente en el dataset transformers_metrics
        fine_metrics.loc[idx, cols_base] = [res[m] for m in cols_base]

flag_fine_metrics=False → se inicia el ponderado


In [112]:
if flag_fine_metrics == False:
  save_metrics(fine_metrics,"5_5_fine_tuning","1_fine_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

Métricas guardadas en /content/drive/MyDrive/neural_profit/5_transformer_model/5_5_fine_tuning/1_fine_metrics.parquet


## **12. Comparación de resultados: Baseline vs Coarse Tuning**


In [113]:
baseline_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
baseline_fold_1,0.002787,0.001900,0.736192,88.063890,0.819911
baseline_fold_2,0.002880,0.001943,0.705611,88.411426,0.812650
baseline_fold_3,0.002713,0.001873,0.719843,86.479732,0.816105
baseline_fold_4,0.002661,0.001812,0.713193,85.306163,0.817494
baseline_fold_5,0.002122,0.001487,0.804769,77.046931,0.839578


In [114]:
coarse_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
coarse_fold_1,0.002441,0.001658,0.788493,80.458990,0.835123
coarse_fold_2,0.002228,0.001550,0.801173,80.309904,0.841547
coarse_fold_3,0.002378,0.001658,0.778362,82.433890,0.831147
coarse_fold_4,0.002341,0.001620,0.773909,81.408258,0.836324
coarse_fold_5,0.002097,0.001470,0.809630,78.728637,0.840471


In [115]:
fine_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
fine_fold_1,0.002561,0.001751,0.771487,83.601671,0.828395
fine_fold_2,0.002992,0.002019,0.68475,88.309174,0.811199
fine_fold_3,0.002109,0.001473,0.81915,76.872899,0.843192
fine_fold_4,0.002751,0.001852,0.693185,87.018496,0.820179
fine_fold_5,0.002026,0.0014,0.819235,76.893788,0.846447


Al comparar el modelo **baseline** con el modelo obtenido tras el **coarse tuning**, se observa una mejora consistente tanto a nivel de folds individuales como en los promedios globales. Promedios sobre 5 folds:

| Métrica | Baseline (avg) | Coarse (avg) | Δ Coarse - Baseline |
|--------|----------------|--------------|---------------------|
| RMSE   | 0.002633 | 0.002297 | **-0.000336** (**-12.75%**) |
| MAE    | 0.001803 | 0.001591 | **-0.000212** (**-11.75%**) |
| R²     | 0.735922 | 0.790313 | **+0.054392** |
| SMAPE  | 85.0616  | 80.6679  | **-4.3937** |
| DirAcc | 0.821148 | 0.836922 | **+0.015775** |



### **12.1. Análisis por fold**

- **Folds 1 a 4**: el modelo coarse supera al baseline en todas las métricas clave (**RMSE, MAE, R², SMAPE y Directional Accuracy**).

- **Fold 5**: se observa una mejora leve en **RMSE, MAE, R² y DirAcc**.  
  El **SMAPE** es ligeramente mayor que en el baseline, aunque el **promedio global de SMAPE** sigue siendo inferior.

### **12.2. Conclusión**

El coarse tuning establece un **nuevo baseline mejorado**, caracterizado por:
- menor error (RMSE y MAE),
- mayor capacidad explicativa (R²),
- mejor desempeño direccional (DirAcc).

Con estos resultados, es razonable avanzar hacia un **fine tuning**, explorando ajustes finos alrededor de `coarse_best_params`.